In [9]:
!pip install pandas xlsxwriter requests pyarrow python-dateutil ipywidgets beautifulsoup4 lxml -q

In [10]:
# ══ Clinical Trials: Data Fetcher ══
import os, hashlib, time, warnings, datetime as _dt
from datetime import date, datetime, timedelta
from pathlib import Path
from collections import OrderedDict
import requests, pandas as pd, pyarrow.parquet as pq, pyarrow as pa
warnings.filterwarnings("ignore")

CT_CACHE_DIR = Path("cache/ct")
CT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
API_URL = "https://clinicaltrials.gov/api/v2/studies"

PHASE_ORDER = [
    "Early Phase 1", "Phase 1", "Phase 1/2",
    "Phase 2", "Phase 2/3", "Phase 3", "Phase 4",
]
PHASE_MAP = {
    frozenset({"EARLY_PHASE1"}): "Early Phase 1",
    frozenset({"PHASE1"}):       "Phase 1",
    frozenset({"PHASE1", "PHASE2"}): "Phase 1/2",
    frozenset({"PHASE2"}):       "Phase 2",
    frozenset({"PHASE2", "PHASE3"}): "Phase 2/3",
    frozenset({"PHASE3"}):       "Phase 3",
    frozenset({"PHASE4"}):       "Phase 4",
}

def _cache_path(s, e):
    h = hashlib.md5(f"{s}|{e}".encode()).hexdigest()[:12]
    return CT_CACHE_DIR / f"ct_{h}.parquet"

def clear_ct_cache():
    for f in CT_CACHE_DIR.glob("*.parquet"):
        f.unlink()

def _fetch_chunk(s_str, e_str):
    essie = (
        f"AREA[StartDate]RANGE[{s_str},{e_str}]"
        " AND AREA[StudyType]INTERVENTIONAL"
        " AND AREA[LeadSponsorClass]INDUSTRY"
    )
    studies, page_token = [], None
    while True:
        params = {"format":"json","pageSize":1000,"filter.advanced":essie}
        if page_token:
            params["pageToken"] = page_token
        resp = requests.get(API_URL, params=params, timeout=60)
        if resp.status_code != 200:
            raise RuntimeError(f"API {resp.status_code}: {resp.text[:500]}\nURL: {resp.url}")
        data = resp.json()
        batch = data.get("studies", [])
        if not batch:
            break
        studies.extend(batch)
        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(0.1)
    return studies

def _month_ranges(start_date, end_date):
    cur = start_date.replace(day=1)
    while cur <= end_date:
        m_start = max(cur, start_date)
        next_month = (cur.replace(day=28) + timedelta(days=4)).replace(day=1)
        m_end = min(next_month - timedelta(days=1), end_date)
        yield m_start, m_end
        cur = next_month

def _parse(studies):
    rows = []
    for s in studies:
        proto = s.get("protocolSection", {})
        ident = proto.get("identificationModule", {})
        status_mod = proto.get("statusModule", {})
        sponsor_mod = proto.get("sponsorCollaboratorsModule", {})
        design_mod = proto.get("designModule", {})
        lead = sponsor_mod.get("leadSponsor", {})
        raw_phases = design_mod.get("phases", [])
        phase_key = frozenset(raw_phases) if raw_phases else frozenset()
        phase_label = PHASE_MAP.get(phase_key, "|".join(sorted(raw_phases)) if raw_phases else "N/A")
        sd_info = status_mod.get("startDateStruct", {})
        sd_str = sd_info.get("date", "")
        try:
            start_dt = pd.Timestamp(sd_str + "-01") if len(sd_str) == 7 else (
                pd.Timestamp(sd_str[:10]) if len(sd_str) >= 10 else pd.NaT)
        except Exception:
            start_dt = pd.NaT
        enr = design_mod.get("enrollmentInfo", {})
        rows.append({
            "NCT ID": ident.get("nctId", ""),
            "Title": ident.get("briefTitle", ""),
            "Sponsor": lead.get("name", ""),
            "Funder Type": lead.get("class", ""),
            "Phase": phase_label,
            "Phase (Raw)": "|".join(sorted(raw_phases)),
            "Status": status_mod.get("overallStatus", ""),
            "Enrollment": enr.get("count"),
            "Enrollment Type": enr.get("type", ""),
            "Start Date": start_dt,
            "Study Type": design_mod.get("studyType", ""),
        })
    return pd.DataFrame(rows)

def fetch_trials(start_date, end_date, progress_cb=None):
    s_str = start_date.strftime("%Y-%m-%d")
    e_str = end_date.strftime("%Y-%m-%d")
    cp = _cache_path(s_str, e_str)
    if cp.exists():
        if progress_cb: progress_cb(1.0, "Loaded from cache")
        return pd.read_parquet(cp)
    chunks = list(_month_ranges(start_date, end_date))
    all_studies = []
    for i, (ms, me) in enumerate(chunks):
        if progress_cb:
            progress_cb(i / len(chunks),
                        f"Chunk {i+1}/{len(chunks)}: {ms.strftime('%b %Y')} "
                        f"({len(all_studies):,} trials so far)")
        chunk = _fetch_chunk(ms.strftime("%Y-%m-%d"), me.strftime("%Y-%m-%d"))
        all_studies.extend(chunk)
        if i < len(chunks) - 1:
            time.sleep(0.15)
    df = _parse(all_studies)
    valid_phases = set(PHASE_ORDER)
    df = df[
        (df["Start Date"] >= pd.Timestamp(start_date))
        & (df["Start Date"] <= pd.Timestamp(end_date))
        & (df["Phase"].isin(valid_phases))
    ]
    df = df.drop_duplicates(subset=["NCT ID"])
    df.to_parquet(cp, index=False)
    if progress_cb: progress_cb(1.0, f"Done \u2014 {len(df):,} trials cached")
    return df



print("CT fetcher ready")


CT fetcher ready


In [11]:
# ══ Clinical Trials: Excel Builder ══
import calendar

CT_GREY = ["#D9D9D9","#B9B9B9","#999999","#7A7A7A","#5A5A5A","#3A3A3A","#1A1A1A"]
CT_BLUE = ["#BDD7EE","#A3BCD7","#89A2C0","#6E87A9","#546D92","#3A527B","#1F3864"]
CT_LABEL_CLR = ["#1A1A1A","#1A1A1A","#1A1A1A","#FFFFFF","#FFFFFF","#FFFFFF","#FFFFFF"]

def _last_complete_month():
    today = date.today()
    return (today.replace(day=1) - timedelta(days=1)).replace(day=1)

def compute_rolling(df):
    anchor = _last_complete_month()
    anchor_end = anchor + pd.DateOffset(months=1) - timedelta(days=1)
    def _c(s, e):
        return int(((df["Start Date"] >= pd.Timestamp(s)) & (df["Start Date"] <= pd.Timestamp(e))).sum())
    def _dim(dt):
        return calendar.monthrange(dt.year, dt.month)[1]
    t12s = anchor - pd.DateOffset(months=11)
    t12 = _c(t12s, anchor_end)
    t12ps = anchor - pd.DateOffset(months=23)
    t12pe = t12s - timedelta(days=1)
    t12p = _c(t12ps, t12pe)
    t12v = ((t12 - t12p) / t12p * 100) if t12p else None
    am_tpd = _c(anchor, anchor_end) / _dim(anchor)
    yrs = anchor - pd.DateOffset(years=1)
    yre = yrs + pd.DateOffset(months=1) - timedelta(days=1)
    yr_tpd = _c(yrs, yre) / _dim(yrs) if _dim(yrs) else 0
    yoy = ((am_tpd - yr_tpd) / yr_tpd * 100) if yr_tpd else None
    ps = anchor - pd.DateOffset(months=1)
    pe = anchor - timedelta(days=1)
    pc_tpd = _c(ps, pe) / _dim(ps) if _dim(ps) else 0
    mom = ((am_tpd - pc_tpd) / pc_tpd * 100) if pc_tpd else None
    p2s = anchor - pd.DateOffset(months=2)
    p2e = ps - timedelta(days=1)
    p2c_tpd = _c(p2s, p2e) / _dim(p2s) if _dim(p2s) else 0
    momp = ((pc_tpd - p2c_tpd) / p2c_tpd * 100) if p2c_tpd else None
    def _w(s, e): return f"{s.strftime('%b %Y')} \u2013 {e.strftime('%b %Y')}"
    return OrderedDict([
        (f"T12M ({_w(t12s, anchor)})", t12),
        (f"T12M Prior ({_w(t12ps, t12pe)})", t12p),
        ("T12M vs T12M Prior %", t12v),
        (f"YoY % trials/day ({anchor.strftime('%b %Y')} vs {yrs.strftime('%b %Y')})", yoy),
        (f"MoM % trials/day ({anchor.strftime('%b %Y')} vs {ps.strftime('%b %Y')})", mom),
        (f"MoM Prior % trials/day ({ps.strftime('%b %Y')} vs {p2s.strftime('%b %Y')})", momp),
    ])

def compute_rolling_enroll(df):
    anchor = _last_complete_month()
    anchor_end = anchor + pd.DateOffset(months=1) - timedelta(days=1)
    def _s(s, e):
        mask = (df["Start Date"] >= pd.Timestamp(s)) & (df["Start Date"] <= pd.Timestamp(e))
        return int(df.loc[mask, "Enrollment"].sum())
    def _dim(dt):
        return calendar.monthrange(dt.year, dt.month)[1]
    t12s = anchor - pd.DateOffset(months=11)
    t12 = _s(t12s, anchor_end)
    t12ps = anchor - pd.DateOffset(months=23)
    t12pe = t12s - timedelta(days=1)
    t12p = _s(t12ps, t12pe)
    t12v = ((t12 - t12p) / t12p * 100) if t12p else None
    am_epd = _s(anchor, anchor_end) / _dim(anchor)
    yrs = anchor - pd.DateOffset(years=1)
    yre = yrs + pd.DateOffset(months=1) - timedelta(days=1)
    yr_epd = _s(yrs, yre) / _dim(yrs) if _dim(yrs) else 0
    yoy = ((am_epd - yr_epd) / yr_epd * 100) if yr_epd else None
    ps = anchor - pd.DateOffset(months=1)
    pe = anchor - timedelta(days=1)
    pc_epd = _s(ps, pe) / _dim(ps) if _dim(ps) else 0
    mom = ((am_epd - pc_epd) / pc_epd * 100) if pc_epd else None
    p2s = anchor - pd.DateOffset(months=2)
    p2e = ps - timedelta(days=1)
    p2c_epd = _s(p2s, p2e) / _dim(p2s) if _dim(p2s) else 0
    momp = ((pc_epd - p2c_epd) / p2c_epd * 100) if p2c_epd else None
    def _w(s, e): return f"{s.strftime('%b %Y')} \u2013 {e.strftime('%b %Y')}"
    return OrderedDict([
        (f"T12M ({_w(t12s, anchor)})", t12),
        (f"T12M Prior ({_w(t12ps, t12pe)})", t12p),
        ("T12M vs T12M Prior %", t12v),
        (f"YoY % enroll/day ({anchor.strftime('%b %Y')} vs {yrs.strftime('%b %Y')})", yoy),
        (f"MoM % enroll/day ({anchor.strftime('%b %Y')} vs {ps.strftime('%b %Y')})", mom),
        (f"MoM Prior % enroll/day ({ps.strftime('%b %Y')} vs {p2s.strftime('%b %Y')})", momp),
    ])

# ─── helper: preview tables for notebook display ─────────────────────────────
def compute_annual(df):
    df2 = df.dropna(subset=["Start Date"]).copy()
    df2["Year"] = df2["Start Date"].dt.year
    pv = df2.pivot_table(index="Year", columns="Phase", values="NCT ID",
                         aggfunc="count", fill_value=0)
    for p in PHASE_ORDER:
        if p not in pv.columns: pv[p] = 0
    pv = pv[PHASE_ORDER]; pv["Total"] = pv.sum(axis=1)
    return pv.sort_index()

def compute_monthly(df, start_date, end_date):
    df2 = df.dropna(subset=["Start Date"]).copy()
    df2["Month"] = df2["Start Date"].values.astype("datetime64[M]")
    s = pd.Timestamp(start_date).replace(day=1)
    e = pd.Timestamp(end_date).replace(day=1)
    mr = pd.date_range(s, e, freq="MS")
    pv = df2.pivot_table(index="Month", columns="Phase", values="NCT ID",
                         aggfunc="count", fill_value=0)
    for p in PHASE_ORDER:
        if p not in pv.columns: pv[p] = 0
    pv = pv[PHASE_ORDER].reindex(mr, fill_value=0)
    pv.index.name = "Month"; pv["Total"] = pv.sum(axis=1)
    return pv

# ─── EXCEL BUILDER ────────────────────────────────────────────────────────────
def build_ct_excel(wb, df, start_date, end_date, ws_dict=None):
    n_data = len(df)
    last_row = n_data + 1  # 1-indexed, header at row 1
    DB = "'Trial Database'"
    DR = f"{DB}!$J$2:$J${last_row}"  # Start Date col
    PR = f"{DB}!$E$2:$E${last_row}"  # Phase col
    ER = f"{DB}!$H$2:$H${last_row}"  # Enrollment col

    # Determine dynamic ranges
    years = list(range(start_date.year, end_date.year + 1))
    month_starts = pd.date_range(
        pd.Timestamp(start_date).replace(day=1),
        pd.Timestamp(end_date).replace(day=1),
        freq="MS"
    )

    # Annualized logic
    today = date.today()
    anchor = _last_complete_month()
    elapsed = anchor.month if anchor.year == today.year else 0
    show_annualized = today.year in years and elapsed > 0

    # Precompute rolling (too complex for Excel formulas)
    rolling = compute_rolling(df)
    enr_rolling = compute_rolling_enroll(df)


    # ── Formats ──
    hdr = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,
        "bg_color":"#1F3864","font_color":"#FFFFFF","border":1,"text_wrap":True,"valign":"vcenter"})
    d0 = wb.add_format({"font_name":"Arial","font_size":9,"border":1})
    d1 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"bg_color":"#F2F2F2"})
    dt0 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"yyyy-mm-dd"})
    dt1 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"yyyy-mm-dd","bg_color":"#F2F2F2"})
    n0 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0"})
    n1 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","bg_color":"#F2F2F2"})
    sc = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","align":"center"})
    st = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","align":"center","bold":True})
    af = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0",
        "align":"center","italic":True,"bg_color":"#FFC000"})
    at_ = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0",
        "align":"center","italic":True,"bold":True,"bg_color":"#FFC000"})
    al_fmt = wb.add_format({"font_name":"Arial","font_size":9,"border":1,
        "italic":True,"bg_color":"#FFC000","bold":True})
    rh = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,
        "bg_color":"#1F3864","font_color":"#FFFFFF","border":1})
    rl = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"bold":True})
    pg = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.0%","align":"center","font_color":"#006100"})
    pr = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.0%","align":"center","font_color":"#C00000"})
    ri = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","align":"center"})
    ml = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"mmm yyyy"})
    dec_fmt = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.00","align":"center"})

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 1 — Trial Database
    # ═══════════════════════════════════════════════════════════════════════════
    ws1 = ws_dict.get("Trial Database", wb.add_worksheet("Trial Database")) if ws_dict else wb.add_worksheet("Trial Database")
    cols = list(df.columns)
    for c, name in enumerate(cols):
        ws1.write(0, c, name, hdr)
    for r, (_, row) in enumerate(df.iterrows(), start=1):
        alt = r % 2 == 0
        for c, col in enumerate(cols):
            val = row[col]
            if pd.isna(val):
                ws1.write_blank(r, c, None, d1 if alt else d0)
            elif col == "Start Date":
                ws1.write_datetime(r, c, val.to_pydatetime(), dt1 if alt else dt0)
            elif col == "Enrollment" and pd.notna(val):
                ws1.write_number(r, c, int(val), n1 if alt else n0)
            else:
                ws1.write(r, c, val, d1 if alt else d0)
    ws1.autofilter(0, 0, n_data, len(cols)-1)
    for i, w in enumerate([14,50,30,14,14,14,18,12,16,12,14]):
        ws1.set_column(i, i, w)
    ws1.freeze_panes(1, 0)

    # ═══════════════════════════════════════════════════════════════════════════
    # Helper: write a formula-based summary tab
    # ═══════════════════════════════════════════════════════════════════════════
    def _write_summary_tab(ws, agg_func, rolling_data, per_day_label):
        # agg_func: "COUNTIFS" or "SUMIFS"
        # For SUMIFS, sum_range is ER; criteria ranges are DR and PR
        ws.set_column(0, 0, 28)
        for i in range(1, 9):
            ws.set_column(i, i, 14)

        # ── Annual (formula-based) ──
        r = 0
        ws.write(r, 0, "Year", hdr)
        for i, p in enumerate(PHASE_ORDER):
            ws.write(r, i+1, p, hdr)
        ws.write(r, 8, "Total", hdr)
        r += 1
        a_start = r
        cur_yr_row = None
        for yr in years:
            ws.write_string(r, 0, str(yr), sc)
            if yr == today.year:
                cur_yr_row = r
            for i, p in enumerate(PHASE_ORDER):
                if agg_func == "COUNTIFS":
                    f_str = f'=COUNTIFS({DR},">="&DATE({yr},1,1),{DR},"<"&DATE({yr+1},1,1),{PR},"{p}")'
                else:
                    f_str = f'=SUMIFS({ER},{DR},">="&DATE({yr},1,1),{DR},"<"&DATE({yr+1},1,1),{PR},"{p}")'
                ws.write_formula(r, i+1, f_str, sc)
            # Total = SUM of phase cols
            row_xl = r + 1
            ws.write_formula(r, 8, f"=SUM(B{row_xl}:H{row_xl})", st)
            r += 1
        a_end_hist = r - 1

        # Annualized row (formula: current year row * 12 / months elapsed)
        ann_chart_row = None
        if show_annualized and cur_yr_row is not None:
            ws.write_string(r, 0, f"{today.year} Annualized", al_fmt)
            yr_xl = cur_yr_row + 1
            for i in range(7):
                col_letter = chr(ord("B") + i)
                ws.write_formula(r, i+1,
                    f"=ROUND({col_letter}{yr_xl}*12/MONTH(EOMONTH(TODAY(),-1)),0)", af)
            row_xl = r + 1
            ws.write_formula(r, 8, f"=SUM(B{row_xl}:H{row_xl})", at_)
            ann_chart_row = r
            r += 1
        a_end = r - 1

        # ── Rolling metrics (computed, not formula) ──
        r += 2
        ws.write(r, 0, "Rolling Metrics", rh)
        ws.write(r, 1, "Value", rh)
        r += 1
        for label, val in rolling_data.items():
            ws.write(r, 0, label, rl)
            if val is None:
                ws.write(r, 1, "N/A", sc)
            elif "%" in label:
                ws.write_number(r, 1, val/100.0, pg if val >= 0 else pr)
            else:
                ws.write_number(r, 1, val, ri)
            r += 1

        # ── Monthly (formula-based, dynamic range) ──
        r += 2
        ws.write(r, 0, "Month", hdr)
        for i, p in enumerate(PHASE_ORDER):
            ws.write(r, i+1, p, hdr)
        ws.write(r, 8, "Total", hdr)
        ws.write(r, 9, "Days", hdr)
        ws.write(r, 10, per_day_label, hdr)
        ws.set_column(9, 9, 8)
        ws.set_column(10, 10, 12)
        r += 1
        m_start = r
        for dt_val in month_starts:
            yr, mo = dt_val.year, dt_val.month
            ws.write_datetime(r, 0, dt_val.to_pydatetime(), ml)
            for i, p in enumerate(PHASE_ORDER):
                if agg_func == "COUNTIFS":
                    f_str = f'=COUNTIFS({DR},">="&DATE({yr},{mo},1),{DR},"<"&DATE({yr},{mo+1},1),{PR},"{p}")'
                else:
                    f_str = f'=SUMIFS({ER},{DR},">="&DATE({yr},{mo},1),{DR},"<"&DATE({yr},{mo+1},1),{PR},"{p}")'
                ws.write_formula(r, i+1, f_str, sc)
            row_xl = r + 1
            ws.write_formula(r, 8, f"=SUM(B{row_xl}:H{row_xl})", st)
            # Days = DAY(EOMONTH(date, 0))
            ws.write_formula(r, 9, f"=DAY(EOMONTH(A{row_xl},0))", sc)
            # Per-day = Total / Days
            ws.write_formula(r, 10, f"=IF(J{row_xl}=0,0,I{row_xl}/J{row_xl})", dec_fmt)
            r += 1
        m_end = r - 1

        return a_start, a_end_hist, a_end, ann_chart_row, m_start, m_end

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 2 — Summary Statistics (Trials)
    # ═══════════════════════════════════════════════════════════════════════════
    SN = "Summary Statistics"
    ws2 = ws_dict.get(SN, wb.add_worksheet(SN)) if ws_dict else wb.add_worksheet(SN)
    a_start, a_end_hist, a_end, ann_chart_row, m_start, m_end = \
        _write_summary_tab(ws2, "COUNTIFS", rolling, "Trials/Day")

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 3 — Summary Statistics - Enrollment
    # ═══════════════════════════════════════════════════════════════════════════
    EN = "Summary Statistics - Enrollment"
    ws_enr = ws_dict.get(EN, wb.add_worksheet(EN)) if ws_dict else wb.add_worksheet(EN)
    ea_start, ea_end_hist, ea_end, enr_ann_chart_row, em_start, em_end = \
        _write_summary_tab(ws_enr, "SUMIFS", enr_rolling, "Enroll/Day")

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 4 — Charts (combo: stacked column + transparent line for totals)
    # ═══════════════════════════════════════════════════════════════════════════
    ws3 = ws_dict.get("CT Charts", wb.add_worksheet("CT Charts")) if ws_dict else wb.add_worksheet("CT Charts")

    def _chart(title, sheet_name, ds, de, pos, is_monthly,
               total_col=8, suppress_bar_idx=None, blue_last_n=1):
        ch = wb.add_chart({"type":"column","subtype":"stacked"})
        ch.set_title({"name":title,"name_font":{"name":"Arial","size":12,"bold":True}})
        ch.set_size({"width":960,"height":520})
        ch.set_chartarea({"border":{"none":True}})
        ch.set_plotarea({"border":{"none":True}})
        ch.set_y_axis({"num_format":"#,##0","major_gridlines":{"visible":False}})
        if is_monthly:
            ch.set_x_axis({"num_format":"mmm yy","label_position":"low",
                "num_font":{"name":"Arial","size":7,"rotation":-45}})
        else:
            ch.set_x_axis({"label_position":"low",
                "num_font":{"name":"Arial","size":8}})

        n_bars = de - ds + 1
        for i, phase in enumerate(PHASE_ORDER):
            col_idx = i + 1
            pts = []
            for bi in range(n_bars):
                g = CT_BLUE if bi >= (n_bars - blue_last_n) else CT_GREY
                pts.append({"fill":{"color":g[i]},"border":{"color":g[i]}})
            dl_cfg = {
                "value":True, "num_format":"#,##0", "position":"center",
                "font":{"name":"Arial","size":7,"color":CT_LABEL_CLR[i]},
            }
            if suppress_bar_idx is not None:
                custom_dl = [None] * n_bars
                custom_dl[suppress_bar_idx] = {"delete": True}
                dl_cfg["custom"] = custom_dl
            ch.add_series({
                "name": phase,
                "categories": [sheet_name, ds, 0, de, 0],
                "values":     [sheet_name, ds, col_idx, de, col_idx],
                "fill":{"color":CT_GREY[i]}, "border":{"color":CT_GREY[i]},
                "points": pts,
                "data_labels": dl_cfg,
                "gap": 80,
            })

        # Transparent line at Total y-value — labels above each bar
        line_ch = wb.add_chart({"type":"line"})
        line_ch.add_series({
            "name": "_Total",
            "categories": [sheet_name, ds, 0, de, 0],
            "values":     [sheet_name, ds, total_col, de, total_col],
            "line":{"none":True},
            "marker":{"type":"none"},
            "data_labels":{
                "value":True,
                "num_format":"#,##0",
                "position":"above",
                "font":{"name":"Arial","size":8,"bold":True,"color":"#1F3864"},
            },
        })
        ch.combine(line_ch)
        ch.set_legend({
            "position":"bottom",
            "font":{"name":"Arial","size":8},
            "delete_series":[len(PHASE_ORDER)],
        })
        ws3.insert_chart(pos, ch)

    # Trials charts
    cur_yr_suppress = None
    blue_n = 1
    if ann_chart_row is not None:
        cur_yr_suppress = a_end_hist - a_start
        blue_n = 2
    _chart("Number of clinical trials by phase, to date: Yearly",
           SN, a_start, a_end, "A1", False,
           suppress_bar_idx=cur_yr_suppress, blue_last_n=blue_n)
    _chart("Number of clinical trials by phase, to date: Monthly",
           SN, m_start, m_end, "A28", True)

    # Enrollment charts
    enr_suppress = None
    enr_blue_n = 1
    if enr_ann_chart_row is not None:
        enr_suppress = ea_end_hist - ea_start
        enr_blue_n = 2
    _chart("Number of enrollment by phase, to date: Yearly",
           EN, ea_start, ea_end, "A55", False,
           suppress_bar_idx=enr_suppress, blue_last_n=enr_blue_n)
    _chart("Number of enrollment by phase, to date: Monthly",
           EN, em_start, em_end, "A82", True)

    return True

print("Stage 2 ready: build_excel()")

print("CT builder ready")


Stage 2 ready: build_excel()
CT builder ready


In [12]:
# ══ FDA Approvals: Data Fetcher ══
import os, hashlib, time, re, warnings, calendar
from datetime import date, datetime, timedelta
from pathlib import Path
from collections import OrderedDict
import requests, pandas as pd
from bs4 import BeautifulSoup
warnings.filterwarnings("ignore")

FDA_CACHE = Path("cache/fda")
FDA_CACHE.mkdir(exist_ok=True)

def _fda_cache_path(prefix, key):
    h = hashlib.md5(key.encode()).hexdigest()[:12]
    return FDA_CACHE / f"{prefix}_{h}.parquet"

def clear_fda_cache():
    for f in FDA_CACHE.glob("*.parquet"):
        f.unlink()

def _fetch_page(url):
    resp = requests.get(url, timeout=30, headers={"User-Agent":"Mozilla/5.0 (research-tracker)"})
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml")

def _extract_content_date(soup):
    for tag in soup.find_all(string=re.compile(r"Content current as of", re.I)):
        text = tag.strip() if isinstance(tag, str) else tag.get_text(strip=True)
        m = re.search(r"(\d{1,2}/\d{1,2}/\d{4})", text)
        if m: return m.group(1)
    return None

# ─── CDER ─────────────────────────────────────────────────────────────────────
def _scrape_cder_year(year):
    url = f"https://www.fda.gov/drugs/novel-drug-approvals-fda/novel-drug-approvals-{year}"
    soup = _fetch_page(url)
    content_date = _extract_content_date(soup)
    table = soup.find("table")
    if not table: return [], content_date, url
    rows = []
    for tr in table.find_all("tr")[1:]:
        cells = tr.find_all(["td","th"])
        if len(cells) < 4: continue
        drug_name = cells[1].get_text(strip=True)
        active = cells[2].get_text(strip=True)
        date_str = cells[3].get_text(strip=True)
        use = cells[4].get_text(strip=True) if len(cells) > 4 else ""
        try: appr_date = pd.Timestamp(date_str)
        except: continue
        # Extract application number from label PDF URL or Drugs@FDA link
        app_num = None
        for cell in cells[1:5]:
            for a_tag in cell.find_all("a", href=True):
                href = a_tag["href"]
                m = re.search(r"/(\d{6})s\d+", href)
                if m: app_num = m.group(1); break
                m2 = re.search(r"ApplNo=(\d+)", href)
                if m2: app_num = m2.group(1); break
            if app_num: break
        rows.append({"Drug Name":drug_name,"Active Ingredient":active,
                     "Approval Date":appr_date,"FDA-Approved Use":use,
                     "_app_num":app_num,"Source":url})
    return rows, content_date, url

def _scrape_drugsfda_month(year, month):
    """Scrape Drugs@FDA monthly report. Returns dict of app_number -> 'NDA' or 'BLA'."""
    url = (f"https://www.accessdata.fda.gov/scripts/cder/daf/index.cfm?"
           f"event=reportsSearch.process&rptName=0&reportSelectMonth={month}&reportSelectYear={year}")
    soup = _fetch_page(url)
    table = soup.find("table")
    if not table: return {}
    lookup = {}
    for tr in table.find_all("tr")[1:]:
        cells = tr.find_all(["td","th"])
        if len(cells) < 7: continue
        drug_col = cells[1].get_text(strip=True)
        submission = cells[2].get_text(strip=True)
        status = cells[6].get_text(strip=True)
        if submission != "ORIG-1" or status != "Approval": continue
        m = re.search(r"(NDA|BLA)\s*#?\s*(\d+)", drug_col)
        if m:
            lookup[m.group(2)] = m.group(1)
    return lookup

def _build_drugsfda_lookup(start_date, end_date, progress_cb=None):
    """Build app_number -> NDA/BLA lookup from Drugs@FDA monthly reports."""
    lookup = {}
    month_list = pd.date_range(
        pd.Timestamp(start_date).replace(day=1),
        pd.Timestamp(end_date).replace(day=1), freq="MS")
    for i, dt in enumerate(month_list):
        if progress_cb:
            progress_cb(i/len(month_list), f"Drugs@FDA: {dt.strftime('%b %Y')}...")
        try:
            monthly = _scrape_drugsfda_month(dt.year, dt.month)
            lookup.update(monthly)
        except Exception as e:
            print(f"  Warning: Drugs@FDA {dt.strftime('%b %Y')}: {e}")
        time.sleep(0.3)
    return lookup

def fetch_cder(start_date, end_date, progress_cb=None):
    cp = _fda_cache_path("cder", f"{start_date}|{end_date}")
    if cp.exists():
        if progress_cb: progress_cb(1.0, "CDER: loaded from cache")
        return pd.read_parquet(cp), None
    years = list(range(start_date.year, end_date.year + 1))
    all_rows, cder_cd = [], None
    for i, yr in enumerate(years):
        if progress_cb: progress_cb(i/len(years)*0.5, f"CDER: scraping {yr}...")
        try:
            rows, cd, url = _scrape_cder_year(yr)
            all_rows.extend(rows)
            if cd: cder_cd = cd
        except Exception as e: print(f"  Warning: CDER {yr}: {e}")
        time.sleep(0.5)
    df = pd.DataFrame(all_rows)
    if len(df) == 0:
        if progress_cb: progress_cb(1.0, "CDER: no data")
        return df, cder_cd
    df = df[(df["Approval Date"]>=pd.Timestamp(start_date))&(df["Approval Date"]<=pd.Timestamp(end_date))]
    # Build NDA/BLA lookup from Drugs@FDA monthly reports, keyed by application number
    if progress_cb: progress_cb(0.55, "Drugs@FDA: building NDA/BLA lookup...")
    daf_lookup = _build_drugsfda_lookup(start_date, end_date, progress_cb)
    # Join: look up each drug's extracted app number in the Drugs@FDA dict
    df["Approval Type"] = df["_app_num"].map(lambda n: daf_lookup.get(n, "Unknown") if n else "Unknown")
    unknown_count = (df["Approval Type"] == "Unknown").sum()
    if unknown_count > 0:
        print(f"  Note: {unknown_count} drug(s) not matched in Drugs@FDA reports:")
        for _, r in df[df["Approval Type"]=="Unknown"].iterrows():
            print(f"    {r['Drug Name']} (app# {r['_app_num']}, date {r['Approval Date'].strftime('%Y-%m-%d')})")
    df = df.drop(columns=["_app_num"])
    df["FDA Center"] = "CDER"
    df["Month"] = df["Approval Date"].dt.month
    df["Year"] = df["Approval Date"].dt.year
    df["Month & Year"] = df["Approval Date"].dt.strftime("%b %Y")
    df = df[["FDA Center","Approval Date","Approval Type","Drug Name","Active Ingredient",
             "Month","Year","Month & Year","FDA-Approved Use","Source"]]
    df.to_parquet(cp, index=False)
    if progress_cb: progress_cb(1.0, f"CDER: {len(df)} approvals cached")
    return df, cder_cd

# ─── CBER ─────────────────────────────────────────────────────────────────────
CBER_BLOOD_KW = ["blood grouping","anti-","reagent","plasma","albumin (human)",
    "immune globulin","antihemophilic","coagulation factor","thrombin","fibrin sealant"]

def _is_blood_product(name, indication):
    text = (name+" "+indication).lower()
    return any(kw.lower() in text for kw in CBER_BLOOD_KW)

def fetch_cber(start_date, end_date, progress_cb=None):
    cp = _fda_cache_path("cber", f"{start_date}|{end_date}")
    if cp.exists():
        if progress_cb: progress_cb(1.0, "CBER: loaded from cache")
        return pd.read_parquet(cp), None
    years = list(range(start_date.year, end_date.year + 1))
    all_rows, cber_cd = [], None
    for i, yr in enumerate(years):
        if progress_cb: progress_cb(i/len(years), f"CBER: scraping {yr}...")
        urls = [
            f"https://www.fda.gov/vaccines-blood-biologics/development-approval-process-cber/{yr}-biological-license-application-approvals",
            f"https://www.fda.gov/vaccines-blood-biologics/development-approval-process-cber/{yr}-biological-approvals",
        ]
        for url in urls:
            try:
                soup = _fetch_page(url)
                cd = _extract_content_date(soup)
                if cd: cber_cd = cd
                table = soup.find("table")
                if not table: continue
                for tr in table.find_all("tr")[1:]:
                    cells = tr.find_all(["td","th"])
                    if len(cells) < 5: continue
                    name = cells[0].get_text(strip=True)
                    indication = cells[1].get_text(strip=True)
                    date_str = cells[-1].get_text(strip=True)
                    if _is_blood_product(name, indication): continue
                    try: appr_date = pd.Timestamp(date_str)
                    except: continue
                    all_rows.append({"FDA Center":"CBER","Approval Date":appr_date,
                        "Approval Type":"BLA","Drug Name":name.split("\n")[0].strip(),
                        "Active Ingredient":name,"FDA-Approved Use":indication,"Source":url})
                break
            except: continue
        time.sleep(0.5)
    df = pd.DataFrame(all_rows)
    if len(df) == 0:
        if progress_cb: progress_cb(1.0, "CBER: no data")
        return df, cber_cd
    df = df[(df["Approval Date"]>=pd.Timestamp(start_date))&(df["Approval Date"]<=pd.Timestamp(end_date))]
    df["Month"] = df["Approval Date"].dt.month
    df["Year"] = df["Approval Date"].dt.year
    df["Month & Year"] = df["Approval Date"].dt.strftime("%b %Y")
    df = df[["FDA Center","Approval Date","Approval Type","Drug Name","Active Ingredient",
             "Month","Year","Month & Year","FDA-Approved Use","Source"]]
    df.to_parquet(cp, index=False)
    if progress_cb: progress_cb(1.0, f"CBER: {len(df)} approvals (as of {cber_cd})")
    return df, cber_cd

# ─── BIOSIMILARS ──────────────────────────────────────────────────────────────
def fetch_biosimilars(start_date, end_date, progress_cb=None):
    cp = _fda_cache_path("biosim", f"{start_date}|{end_date}")
    if cp.exists():
        if progress_cb: progress_cb(1.0, "Biosimilars: loaded from cache")
        return pd.read_parquet(cp), None
    if progress_cb: progress_cb(0.2, "Biosimilars: scraping...")
    url = "https://www.fda.gov/drugs/biosimilars/biosimilar-product-information"
    soup = _fetch_page(url)
    content_date = _extract_content_date(soup)
    table = soup.find("table")
    if not table:
        if progress_cb: progress_cb(1.0, "Biosimilars: no table")
        return pd.DataFrame(), content_date
    rows = []
    for tr in table.find_all("tr")[1:]:
        cells = tr.find_all(["td","th"])
        if len(cells) < 3: continue
        biosim_name = cells[0].get_text(strip=True)
        date_str = cells[1].get_text(strip=True)
        ref_product = cells[2].get_text(strip=True)
        if not biosim_name or not date_str: continue
        try: appr_date = pd.Timestamp(date_str)
        except:
            try: appr_date = pd.Timestamp(f"01 {date_str}")
            except: continue
        count = len(re.split(r"\s+and\s+", ref_product))
        ai_match = re.search(r"\(([^)]+)\)", biosim_name)
        rows.append({"Approval Date":date_str,"Approval Year":appr_date.year,
            "Approval Month & Year":appr_date.strftime("%b %Y"),
            "Biosimilar Name":biosim_name,
            "Active Ingredient":ai_match.group(1) if ai_match else "",
            "Reference Product":ref_product,"Count":count,"Source":url,
            "_parsed_date":appr_date})
    df = pd.DataFrame(rows)
    if len(df) == 0:
        if progress_cb: progress_cb(1.0, "Biosimilars: no data")
        return df, content_date
    df = df[(df["_parsed_date"]>=pd.Timestamp(start_date))&(df["_parsed_date"]<=pd.Timestamp(end_date))]
    df = df.drop(columns=["_parsed_date"])
    df.to_parquet(cp, index=False)
    if progress_cb: progress_cb(1.0, f"Biosimilars: {len(df)} entries (as of {content_date})")
    return df, content_date

# ─── COMBINED ─────────────────────────────────────────────────────────────────
def fetch_all_fda(start_date, end_date, progress_cb=None):
    def _sub(stage, total):
        def cb(pct, msg):
            if progress_cb: progress_cb((stage+pct)/total, msg)
        return cb
    cder, cder_date = fetch_cder(start_date, end_date, _sub(0,3))
    cber, cber_date = fetch_cber(start_date, end_date, _sub(1,3))
    biosim, biosim_date = fetch_biosimilars(start_date, end_date, _sub(2,3))
    novel = pd.concat([cder, cber], ignore_index=True)
    if len(novel) > 0:
        novel = novel.sort_values("Approval Date", ascending=False).reset_index(drop=True)
    if progress_cb:
        progress_cb(1.0, f"Done: {len(novel)} novel, {len(biosim)} biosimilars")
    return novel, biosim, cder_date, cber_date, biosim_date



print("FDA fetcher ready")


FDA fetcher ready


In [13]:
# ══ FDA Approvals: Excel Builder ══

BLACK = "#1A1A1A"; GREY = "#999999"; LIGHT_GREY = "#D9D9D9"
DARK_BLUE = "#1F3864"; LIGHT_BLUE = "#BDD7EE"

def _last_complete_month():
    today = date.today()
    return (today.replace(day=1) - timedelta(days=1)).replace(day=1)

def _compute_novel_rolling(novel_df):
    anchor = _last_complete_month()
    anchor_end = anchor + pd.DateOffset(months=1) - timedelta(days=1)
    def _c(s,e):
        return int(((novel_df["Approval Date"]>=pd.Timestamp(s))&(novel_df["Approval Date"]<=pd.Timestamp(e))).sum())
    t12s = anchor - pd.DateOffset(months=11)
    t12 = _c(t12s, anchor_end)
    t12ps = anchor - pd.DateOffset(months=23)
    t12pe = t12s - timedelta(days=1)
    t12p = _c(t12ps, t12pe)
    t12v = ((t12-t12p)/t12p*100) if t12p else None
    # Full year comparison
    prev_yr = anchor.year - 1
    prev2_yr = prev_yr - 1
    yr1 = _c(date(prev_yr,1,1), date(prev_yr,12,31))
    yr2 = _c(date(prev2_yr,1,1), date(prev2_yr,12,31))
    yr_v = ((yr1-yr2)/yr2*100) if yr2 else None
    # Sequential
    am = _c(anchor, anchor_end)
    ps = anchor - pd.DateOffset(months=1)
    pe = anchor - timedelta(days=1)
    pc = _c(ps, pe)
    seq = ((am-pc)/pc*100) if pc else None
    p2s = anchor - pd.DateOffset(months=2)
    p2e = ps - timedelta(days=1)
    p2c = _c(p2s, p2e)
    seq_prev = ((pc-p2c)/p2c*100) if p2c else None
    def _w(s,e): return f"{s.strftime('%b %Y')} \u2013 {e.strftime('%b %Y')}"
    return OrderedDict([
        (f"T12M ({_w(t12s, anchor)})", t12),
        (f"T12M Prior ({_w(t12ps, t12pe)})", t12p),
        ("T12M vs T12M Prior %", t12v),
        (f"{prev_yr} vs {prev2_yr} %", yr_v),
        (f"Latest Mo Seq Growth ({anchor.strftime('%b %Y')} vs {ps.strftime('%b %Y')})", seq),
        (f"Prev Mo Seq Growth ({ps.strftime('%b %Y')} vs {p2s.strftime('%b %Y')})", seq_prev),
    ])

def _compute_biosim_rolling(biosim_df):
    anchor = _last_complete_month()
    anchor_end = anchor + pd.DateOffset(months=1) - timedelta(days=1)
    biosim_df = biosim_df.copy()
    if "_parsed_date" not in biosim_df.columns:
        biosim_df["_parsed_date"] = pd.to_datetime(biosim_df["Approval Date"], format="mixed", dayfirst=False)
    def _s(s,e):
        mask = (biosim_df["_parsed_date"]>=pd.Timestamp(s))&(biosim_df["_parsed_date"]<=pd.Timestamp(e))
        return int(biosim_df.loc[mask,"Count"].sum())
    t12s = anchor - pd.DateOffset(months=11)
    t12 = _s(t12s, anchor_end)
    t12ps = anchor - pd.DateOffset(months=23)
    t12pe = t12s - timedelta(days=1)
    t12p = _s(t12ps, t12pe)
    t12v = ((t12-t12p)/t12p*100) if t12p else None
    prev_yr = anchor.year - 1; prev2_yr = prev_yr - 1
    yr1 = _s(date(prev_yr,1,1), date(prev_yr,12,31))
    yr2 = _s(date(prev2_yr,1,1), date(prev2_yr,12,31))
    yr_v = ((yr1-yr2)/yr2*100) if yr2 else None
    am = _s(anchor, anchor_end)
    ps = anchor - pd.DateOffset(months=1)
    pe = anchor - timedelta(days=1)
    pc = _s(ps, pe)
    seq = ((am-pc)/pc*100) if pc else None
    p2s = anchor - pd.DateOffset(months=2)
    p2e = ps - timedelta(days=1)
    p2c = _s(p2s, p2e)
    seq_prev = ((pc-p2c)/p2c*100) if p2c else None
    def _w(s,e): return f"{s.strftime('%b %Y')} \u2013 {e.strftime('%b %Y')}"
    return OrderedDict([
        (f"T12M ({_w(t12s, anchor)})", t12),
        (f"T12M Prior ({_w(t12ps, t12pe)})", t12p),
        ("T12M vs T12M Prior %", t12v),
        (f"{prev_yr} vs {prev2_yr} %", yr_v),
        (f"Latest Mo Seq Growth ({anchor.strftime('%b %Y')} vs {ps.strftime('%b %Y')})", seq),
        (f"Prev Mo Seq Growth ({ps.strftime('%b %Y')} vs {p2s.strftime('%b %Y')})", seq_prev),
    ])

def build_fda_excel(wb, novel_df, biosim_df, start_date, end_date,
                    cder_date, cber_date, biosim_date, ws_dict=None):
    pull_date = date.today().strftime("%m/%d/%Y")
    n_novel = len(novel_df)
    n_biosim = len(biosim_df)
    last_novel = n_novel + 1
    last_biosim = n_biosim + 1
    DB_N = "'Novel Approval Database'"
    DB_B = "'Biosimilar Approval Database'"
    # Novel column refs (0-indexed): B=ApprDate, C=ApprType, D=DrugName, A=Center, G=Year, H=M&Y
    # Excel 1-indexed: A=1(Center), B=2(Date), C=3(Type), G=7(Year), H=8(M&Y)
    CENTER_R = f"{DB_N}!$A$3:$A${last_novel+1}"
    TYPE_R = f"{DB_N}!$C$3:$C${last_novel+1}"
    DATE_R = f"{DB_N}!$B$3:$B${last_novel+1}"
    MY_R = f"{DB_N}!$H$3:$H${last_novel+1}"
    YEAR_R = f"{DB_N}!$G$3:$G${last_novel+1}"
    # Biosim refs: A=Date, B=Year, C=M&Y, G=Count
    BIOSIM_MY = f"{DB_B}!$C$3:$C${last_biosim+1}"
    BIOSIM_YR = f"{DB_B}!$B$3:$B${last_biosim+1}"
    BIOSIM_CT = f"{DB_B}!$G$3:$G${last_biosim+1}"

    years = list(range(start_date.year, end_date.year + 1))
    month_starts = pd.date_range(pd.Timestamp(start_date).replace(day=1),
                                  pd.Timestamp(end_date).replace(day=1), freq="MS")
    today = date.today()
    anchor = _last_complete_month()
    elapsed = anchor.month if anchor.year == today.year else 0
    show_ann = today.year in years and elapsed > 0

    novel_rolling = _compute_novel_rolling(novel_df) if n_novel > 0 else {}
    biosim_rolling = _compute_biosim_rolling(biosim_df) if n_biosim > 0 else {}


    # ── Formats ──
    hdr = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,"bg_color":"#1F3864",
        "font_color":"#FFFFFF","border":1,"text_wrap":True,"valign":"vcenter","align":"center"})
    hdr_l = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,"bg_color":"#1F3864",
        "font_color":"#FFFFFF","border":1,"text_wrap":True,"valign":"vcenter"})
    d0 = wb.add_format({"font_name":"Arial","font_size":9,"border":1})
    d1 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"bg_color":"#F2F2F2"})
    dt0 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"mm/dd/yyyy"})
    dt1 = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"mm/dd/yyyy","bg_color":"#F2F2F2"})
    sc = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","align":"center"})
    st = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","align":"center","bold":True})
    af = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0",
        "align":"center","italic":True,"bg_color":"#FFC000"})
    at_ = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0",
        "align":"center","italic":True,"bold":True,"bg_color":"#FFC000"})
    al = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"italic":True,"bg_color":"#FFC000","bold":True})
    pull_f = wb.add_format({"font_name":"Arial","font_size":9,"italic":True,"font_color":"#666666"})
    bl = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"bold":True})
    rh = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,"bg_color":"#1F3864",
        "font_color":"#FFFFFF","border":1})
    rl = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"bold":True})
    ri = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","align":"center"})
    dec = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.00","align":"center"})
    pg = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.0%","align":"center","font_color":"#006100"})
    pr = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.0%","align":"center","font_color":"#C00000"})
    sec = wb.add_format({"bold":True,"font_name":"Arial","font_size":11,"bottom":2,"bottom_color":"#1F3864"})
    note_f = wb.add_format({"font_name":"Arial","font_size":8,"italic":True,"font_color":"#666666"})
    ml = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"mmm yyyy","align":"center"})

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 1 — Novel Approval Database
    # ═══════════════════════════════════════════════════════════════════════════
    ws1 = ws_dict.get("Novel Approval Database", wb.add_worksheet("Novel Approval Database")) if ws_dict else wb.add_worksheet("Novel Approval Database")
    ws1.write(0, 0, f"Data pulled: {pull_date} | CDER current as of: {cder_date or 'N/A'} | CBER current as of: {cber_date or 'N/A'}", pull_f)
    cols1 = ["FDA Center","Approval Date","Approval Type","Drug Name","Active Ingredient",
             "Month","Year","Month & Year","FDA-Approved Use","Source"]
    for c, name in enumerate(cols1):
        ws1.write(1, c, name, hdr_l if c >= 3 else hdr)
    for r, (_, row) in enumerate(novel_df.iterrows(), start=2):
        alt = r % 2 == 1
        for c, col in enumerate(cols1):
            val = row.get(col)
            if col == "Approval Date" and pd.notna(val):
                ws1.write_datetime(r, c, val.to_pydatetime(), dt1 if alt else dt0)
            elif col in ("Month","Year") and pd.notna(val):
                ws1.write_number(r, c, int(val), sc)
            else:
                ws1.write(r, c, val if pd.notna(val) else "", d1 if alt else d0)
    ws1.autofilter(1, 0, n_novel+1, len(cols1)-1)
    for i, w in enumerate([12,14,14,18,22,8,8,12,40,50]):
        ws1.set_column(i, i, w)
    ws1.freeze_panes(2, 0)

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 2 — Biosimilar Approval Database
    # ═══════════════════════════════════════════════════════════════════════════
    ws2 = ws_dict.get("Biosimilar Approval Database", wb.add_worksheet("Biosimilar Approval Database")) if ws_dict else wb.add_worksheet("Biosimilar Approval Database")
    ws2.write(0, 0, f"Data pulled: {pull_date} | Biosimilar page current as of: {biosim_date or 'N/A'}", pull_f)
    cols2 = ["Approval Date","Approval Year","Approval Month & Year","Biosimilar Name",
             "Active Ingredient","Reference Product","Count","Source"]
    for c, name in enumerate(cols2):
        ws2.write(1, c, name, hdr_l if c >= 3 else hdr)
    for r, (_, row) in enumerate(biosim_df.iterrows(), start=2):
        alt = r % 2 == 1
        for c, col in enumerate(cols2):
            val = row.get(col)
            if col in ("Approval Year","Count") and pd.notna(val):
                ws2.write_number(r, c, int(val), sc)
            else:
                ws2.write(r, c, val if pd.notna(val) else "", d1 if alt else d0)
    ws2.autofilter(1, 0, n_biosim+1, len(cols2)-1)
    for i, w in enumerate([14,12,18,40,20,35,8,50]):
        ws2.set_column(i, i, w)
    ws2.freeze_panes(2, 0)

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 3 — Approval Summary
    # ═══════════════════════════════════════════════════════════════════════════
    ws3 = ws_dict.get("FDA Approval Summary", wb.add_worksheet("FDA Approval Summary")) if ws_dict else wb.add_worksheet("FDA Approval Summary")
    ws3.set_column(0, 0, 38)
    for i in range(1, 12):
        ws3.set_column(i, i, 14)
    ws3.write(0, 0, f"Data pulled: {pull_date}", pull_f)

    # ── A) Annual Novel ──
    r = 2
    ws3.write(r, 0, "A) Annual Novel Approval Summary", sec); r += 1
    ws3.write(r, 0, "Year", hdr); ws3.write(r, 1, "Chemical (NDA)", hdr)
    ws3.write(r, 2, "Biologics (BLA)", hdr); ws3.write(r, 3, "Total", hdr); r += 1
    a_start = r
    cur_yr_row = None
    for yr in years:
        ws3.write_string(r, 0, str(yr), sc)
        if yr == today.year: cur_yr_row = r
        ws3.write_formula(r, 1, f'=COUNTIFS({YEAR_R},{yr},{TYPE_R},"NDA")', sc)
        ws3.write_formula(r, 2, f'=COUNTIFS({YEAR_R},{yr},{TYPE_R},"BLA")', sc)
        rn = r + 1
        ws3.write_formula(r, 3, f"=B{rn}+C{rn}", st)
        r += 1
    a_end_hist = r - 1
    if show_ann and cur_yr_row is not None:
        ws3.write_string(r, 0, f"{today.year} Annualized", al)
        yr_xl = cur_yr_row + 1
        ws3.write_formula(r, 1, f"=ROUND(B{yr_xl}*12/MONTH(EOMONTH(TODAY(),-1)),0)", af)
        ws3.write_formula(r, 2, f"=ROUND(C{yr_xl}*12/MONTH(EOMONTH(TODAY(),-1)),0)", af)
        rn = r + 1
        ws3.write_formula(r, 3, f"=B{rn}+C{rn}", at_)
        r += 1
    a_end = r - 1

    # ── B) Monthly Novel ──
    r += 1; ws3.write(r, 0, "B) Monthly Novel Approval Summary", sec); r += 1
    ws3.write(r, 0, "Month", hdr)
    ws3.write(r, 1, "NDA", hdr); ws3.write(r, 2, "BLA (CDER)", hdr)
    ws3.write(r, 3, f"BLA (CBER)*", hdr)
    ws3.write(r, 4, "Total", hdr); ws3.write(r, 5, "Days", hdr)
    ws3.write(r, 6, "Approvals/Day", hdr)
    ws3.write(r, 7, "Seq Growth %", hdr)
    ws3.write(r, 8, "CDER #", hdr)
    ws3.write(r, 9, "YoY CDER %", hdr); r += 1
    m_start = r
    for j, dt_val in enumerate(month_starts):
        yr, mo = dt_val.year, dt_val.month
        my_str = dt_val.strftime("%b %Y")
        ws3.write_datetime(r, 0, dt_val.to_pydatetime(), ml)
        ws3.write_formula(r, 1, f'=COUNTIFS({MY_R},"{my_str}",{TYPE_R},"NDA")', sc)
        ws3.write_formula(r, 2, f'=COUNTIFS({MY_R},"{my_str}",{TYPE_R},"BLA",{CENTER_R},"CDER")', sc)
        ws3.write_formula(r, 3, f'=COUNTIFS({MY_R},"{my_str}",{CENTER_R},"CBER")', sc)
        rn = r + 1
        ws3.write_formula(r, 4, f"=B{rn}+C{rn}+D{rn}", st)
        ws3.write_formula(r, 5, f"=DAY(EOMONTH(A{rn},0))", sc)
        ws3.write_formula(r, 6, f"=IF(F{rn}=0,0,E{rn}/F{rn})", dec)
        if j > 0:
            ws3.write_formula(r, 7, f'=IF(E{rn-1}=0,"N/A",(E{rn}-E{rn-1})/E{rn-1})', pg)
        else:
            ws3.write(r, 7, "N/A", sc)
        ws3.write_formula(r, 8, f"=B{rn}+C{rn}", sc)
        # YoY: find same month prior year
        prior_my = (dt_val - pd.DateOffset(years=1)).strftime("%b %Y")
        ws3.write_formula(r, 9,
            f'=IFERROR((I{rn}-COUNTIFS({MY_R},"{prior_my}",{TYPE_R},"NDA")-COUNTIFS({MY_R},"{prior_my}",{TYPE_R},"BLA",{CENTER_R},"CDER"))'
            f'/(COUNTIFS({MY_R},"{prior_my}",{TYPE_R},"NDA")+COUNTIFS({MY_R},"{prior_my}",{TYPE_R},"BLA",{CENTER_R},"CDER")),"N/A")', pg)
        r += 1
    m_end = r - 1
    ws3.write(r, 0, f"*CBER data current as of: {cber_date or 'N/A'}", note_f); r += 1

    # ── C) Annual Biosimilar ──
    r += 1; ws3.write(r, 0, "C) Annual Biosimilar Approval Summary", sec); r += 1
    ws3.write(r, 0, "Year", hdr); ws3.write(r, 1, "Approvals", hdr); r += 1
    c_start = r
    c_cur_yr_row = None
    for yr in years:
        ws3.write_string(r, 0, str(yr), sc)
        if yr == today.year: c_cur_yr_row = r
        ws3.write_formula(r, 1, f"=SUMIFS({BIOSIM_CT},{BIOSIM_YR},{yr})", sc)
        r += 1
    c_end_hist = r - 1
    if show_ann and c_cur_yr_row is not None:
        ws3.write_string(r, 0, f"{today.year} Annualized", al)
        yr_xl = c_cur_yr_row + 1
        ws3.write_formula(r, 1, f"=ROUND(B{yr_xl}*12/MONTH(EOMONTH(TODAY(),-1)),0)", af)
        r += 1
    c_end = r - 1

    # ── D) Monthly Biosimilar ──
    r += 1; ws3.write(r, 0, "D) Monthly Biosimilar Approval Summary", sec); r += 1
    ws3.write(r, 0, "Month", hdr); ws3.write(r, 1, "Approvals", hdr)
    ws3.write(r, 2, "Days", hdr); ws3.write(r, 3, "Approvals/Day", hdr)
    ws3.write(r, 4, "Seq Growth %", hdr); ws3.write(r, 5, "YoY Growth %", hdr); r += 1
    d_start = r
    for j, dt_val in enumerate(month_starts):
        my_str = dt_val.strftime("%b %Y")
        ws3.write_datetime(r, 0, dt_val.to_pydatetime(), ml)
        ws3.write_formula(r, 1, f'=SUMIFS({BIOSIM_CT},{BIOSIM_MY},"{my_str}")', sc)
        rn = r + 1
        ws3.write_formula(r, 2, f"=DAY(EOMONTH(A{rn},0))", sc)
        ws3.write_formula(r, 3, f"=IF(C{rn}=0,0,B{rn}/C{rn})", dec)
        if j > 0:
            ws3.write_formula(r, 4, f'=IF(B{rn-1}=0,"N/A",(B{rn}-B{rn-1})/B{rn-1})', pg)
        else:
            ws3.write(r, 4, "N/A", sc)
        prior_my = (dt_val - pd.DateOffset(years=1)).strftime("%b %Y")
        ws3.write_formula(r, 5, f'=IFERROR((B{rn}-SUMIFS({BIOSIM_CT},{BIOSIM_MY},"{prior_my}"))/SUMIFS({BIOSIM_CT},{BIOSIM_MY},"{prior_my}"),"N/A")', pg)
        r += 1
    d_end = r - 1

    # ── E) Overall Novel Summary ──
    r += 1; ws3.write(r, 0, "E) Overall Novel Approval Summary", sec); r += 1
    ws3.write(r, 0, "Metric", rh); ws3.write(r, 1, "Value", rh); r += 1
    for label, val in novel_rolling.items():
        ws3.write(r, 0, label, rl)
        if val is None: ws3.write(r, 1, "N/A", sc)
        elif "%" in label: ws3.write_number(r, 1, val/100.0, pg if val >= 0 else pr)
        else: ws3.write_number(r, 1, val, ri)
        r += 1

    # ── F) Overall Biosimilar Summary ──
    r += 1; ws3.write(r, 0, "F) Overall Biosimilar Approval Summary", sec); r += 1
    ws3.write(r, 0, "Metric", rh); ws3.write(r, 1, "Value", rh); r += 1
    for label, val in biosim_rolling.items():
        ws3.write(r, 0, label, rl)
        if val is None: ws3.write(r, 1, "N/A", sc)
        elif "%" in label: ws3.write_number(r, 1, val/100.0, pg if val >= 0 else pr)
        else: ws3.write_number(r, 1, val, ri)
        r += 1

    # ═══════════════════════════════════════════════════════════════════════════
    # TAB 4 — Charts
    # ═══════════════════════════════════════════════════════════════════════════
    ws4 = ws_dict.get("FDA Charts", wb.add_worksheet("FDA Charts")) if ws_dict else wb.add_worksheet("FDA Charts")
    SN = "FDA Approval Summary"

    def _novel_chart(title, ds, de, pos, is_monthly, total_col,
                     series_info, suppress_idx=None, blue_last_n=1):
        ch = wb.add_chart({"type":"column","subtype":"stacked"})
        ch.set_title({"name":title,"name_font":{"name":"Arial","size":12,"bold":True}})
        ch.set_size({"width":960,"height":520})
        ch.set_chartarea({"border":{"none":True}})
        ch.set_plotarea({"border":{"none":True}})
        ch.set_y_axis({"num_format":"#,##0","major_gridlines":{"visible":False}})
        if is_monthly:
            ch.set_x_axis({"num_format":"mmm yy","label_position":"low",
                "num_font":{"name":"Arial","size":7,"rotation":-45}})
        else:
            ch.set_x_axis({"label_position":"low","num_font":{"name":"Arial","size":8}})
        n_bars = de - ds + 1
        for si, (sname, col_idx, hist_color, blue_color, lbl_color) in enumerate(series_info):
            pts = []
            for bi in range(n_bars):
                c = blue_color if bi >= n_bars - blue_last_n else hist_color
                pts.append({"fill":{"color":c},"border":{"color":c}})
            dl = {"value":True,"num_format":"#,##0","position":"center",
                  "font":{"name":"Arial","size":8,"color":lbl_color}}
            if suppress_idx is not None:
                custom = [None]*n_bars; custom[suppress_idx] = {"delete":True}
                dl["custom"] = custom
            ch.add_series({"name":sname,"categories":[SN,ds,0,de,0],
                "values":[SN,ds,col_idx,de,col_idx],
                "fill":{"color":hist_color},"border":{"color":hist_color},
                "points":pts,"data_labels":dl,"gap":80})
        line = wb.add_chart({"type":"line"})
        line.add_series({"name":"_Total","categories":[SN,ds,0,de,0],
            "values":[SN,ds,total_col,de,total_col],
            "line":{"none":True},"marker":{"type":"none"},
            "data_labels":{"value":True,"num_format":"#,##0","position":"above",
                "font":{"name":"Arial","size":9,"bold":True,"color":DARK_BLUE}}})
        ch.combine(line)
        ch.set_legend({"position":"bottom","font":{"name":"Arial","size":8},
            "delete_series":[len(series_info)]})
        ws4.insert_chart(pos, ch)

    def _single_chart(title, sheet, ds, de, pos, val_col, blue_last_n=1):
        ch = wb.add_chart({"type":"column"})
        ch.set_title({"name":title,"name_font":{"name":"Arial","size":12,"bold":True}})
        ch.set_size({"width":960,"height":520})
        ch.set_chartarea({"border":{"none":True}})
        ch.set_plotarea({"border":{"none":True}})
        ch.set_y_axis({"num_format":"#,##0","major_gridlines":{"visible":False}})
        ch.set_x_axis({"label_position":"low","num_font":{"name":"Arial","size":8}})
        n = de - ds + 1
        pts = []
        for i in range(n):
            c = LIGHT_BLUE if i >= n - blue_last_n else GREY
            pts.append({"fill":{"color":c},"border":{"color":c}})
        ch.add_series({"name":"Approvals","categories":[sheet,ds,0,de,0],
            "values":[sheet,ds,val_col,de,val_col],
            "fill":{"color":GREY},"border":{"color":GREY},"points":pts,
            "data_labels":{"value":True,"num_format":"#,##0","position":"outside_end",
                "font":{"name":"Arial","size":9,"bold":True,"color":DARK_BLUE}},
            "gap":80})
        ch.set_legend({"none":True})
        ws4.insert_chart(pos, ch)

    # i) Novel Yearly
    suppress = None; blue_n = 1
    if show_ann and cur_yr_row is not None:
        suppress = a_end_hist - a_start
        blue_n = 2
    _novel_chart("FDA new drug approvals, to date: Yearly",
        a_start, a_end, "A1", False, total_col=3,
        series_info=[
            ("Chemical (NDA)", 1, GREY, LIGHT_BLUE, "#FFFFFF"),
            ("Biologics (BLA)", 2, BLACK, DARK_BLUE, "#FFFFFF"),
        ], suppress_idx=suppress, blue_last_n=blue_n)

    # ii) Novel Monthly
    cber_note = f"\n(CBER data as of {cber_date})" if cber_date else ""
    _novel_chart(f"FDA new drug approvals, to date: Monthly{cber_note}",
        m_start, m_end, "A28", True, total_col=4,
        series_info=[
            ("NDA", 1, LIGHT_GREY, LIGHT_BLUE, "#1A1A1A"),
            ("BLA (CDER)", 2, GREY, DARK_BLUE, "#FFFFFF"),
            ("BLA (CBER)", 3, BLACK, "#0D3B5E", "#FFFFFF"),
        ], blue_last_n=1)

    # iii) Biosimilar Yearly
    c_blue = 1; c_suppress = None
    if show_ann and c_cur_yr_row is not None:
        c_blue = 2
    _single_chart("FDA biosimilar approvals, to date: Yearly",
        SN, c_start, c_end, "A55", val_col=1, blue_last_n=c_blue)

    # iv) Biosimilar Monthly
    _single_chart("FDA biosimilar approvals, to date: Monthly",
        SN, d_start, d_end, "A82", val_col=1, blue_last_n=1)

    return True

print("Stage 2 ready: build_fda_excel()")

print("FDA builder ready")


Stage 2 ready: build_fda_excel()
FDA builder ready


In [14]:
# ══ NIH Outlays: Data Fetcher ══
import requests, pandas as pd, hashlib, shutil
from pathlib import Path
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

MTS_CACHE_DIR = Path("cache/nih")
MTS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MTS_BASE_URL = "https://api.fiscaldata.treasury.gov/services/api/fiscal_service"
MTS_TABLE = "mts_table_5"
MTS_PULL_ALL_HHS = True


def fetch_mts_outlays(start_date, end_date, progress_cb=None):
    cache_key = hashlib.md5(f"mts_{start_date}_{end_date}".encode()).hexdigest()[:12]
    cache_path = MTS_CACHE_DIR / f"mts_outlays_{cache_key}.parquet"
    
    if cache_path.exists():
        if progress_cb: progress_cb(1.0, "MTS: loaded from cache")
        print(f"[MTS] Cache hit: {cache_path}")
        return pd.read_parquet(cache_path)
    
    if progress_cb: progress_cb(0.05, "MTS: querying Treasury API...")
    
    all_records, page = [], 1
    page_size = 1000
    endpoint = f"/v1/accounting/mts/{MTS_TABLE}"
    base_filter = f"record_date:gte:{start_date},record_date:lte:{end_date}"
    
    while True:
        url = (f"{MTS_BASE_URL}{endpoint}?filter={base_filter}"
               f"&sort=record_date,classification_id"
               f"&page[number]={page}&page[size]={page_size}&format=json")
        
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        
        if page == 1 and not data.get("data"):
            alt = "mts_table_6" if "table_5" in endpoint else "mts_table_5"
            print(f"[MTS] {MTS_TABLE} empty, trying {alt}...")
            endpoint = f"/v1/accounting/mts/{alt}"
            continue
        
        records = data.get("data", [])
        if not records: break
        
        all_records.extend(records)
        total_count = int(data["meta"]["total-count"])
        
        if progress_cb:
            progress_cb(min(0.05 + 0.85 * len(all_records) / max(total_count, 1), 0.90),
                        f"MTS: {len(all_records)}/{total_count}")
        
        if len(all_records) >= total_count: break
        page += 1
    
    if not all_records:
        print("[MTS] WARNING: No records returned.")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_records)
    for col in [c for c in df.columns if "amt" in c.lower()]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    hhs_mask = df["classification_desc"].str.contains("Health and Human Services", case=False, na=False)
    nih_mask = df["classification_desc"].str.contains("National Institutes of Health", case=False, na=False)
    
    if MTS_PULL_ALL_HHS and "parent_id" in df.columns:
        hhs_ids = df.loc[hhs_mask, "classification_id"].unique()
        df_f = df[hhs_mask | df["parent_id"].isin(hhs_ids) | nih_mask].copy()
    else:
        df_f = df[hhs_mask | nih_mask].copy()
    
    if df_f.empty:
        print("[MTS] WARNING: No HHS/NIH rows. Keeping all.")
        df_f = df.copy()
    
    df_f["is_nih"] = df_f["classification_desc"].str.contains("National Institutes of Health", case=False, na=False)
    df_f["is_hhs_total"] = hhs_mask.reindex(df_f.index, fill_value=False) & ~df_f["is_nih"]
    df_f["record_date"] = pd.to_datetime(df_f["record_date"])
    
    for col in ["record_fiscal_year", "record_calendar_year", "record_calendar_month"]:
        if col in df_f.columns:
            df_f[col] = pd.to_numeric(df_f[col], errors="coerce").astype("Int64")
    
    if "parent_id" in df_f.columns:
        plookup = df.drop_duplicates("classification_id").set_index("classification_id")["classification_desc"].to_dict()
        df_f["parent_desc"] = df_f["parent_id"].map(plookup).fillna("")
    else:
        df_f["parent_desc"] = ""
    
    df_f = df_f.sort_values(["record_date", "classification_desc"]).reset_index(drop=True)
    df_f.to_parquet(cache_path, index=False)
    
    print(f"[MTS] {len(df_f)} rows | {df_f['record_date'].min().date()} → {df_f['record_date'].max().date()}")
    if progress_cb: progress_cb(1.0, f"MTS: {len(df_f)} records cached")
    return df_f


def clear_mts_cache():
    if MTS_CACHE_DIR.exists():
        shutil.rmtree(MTS_CACHE_DIR)
        MTS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        print("[MTS] Cache cleared.")

print("NIH fetcher ready")


NIH fetcher ready


In [15]:
# ══ NIH Outlays: Excel Builder ══
import numpy as np
from datetime import timedelta
import calendar


def build_mts_excel(workbook, df_mts, start_date_str, end_date_str, ws_dict=None):
    if df_mts is None or df_mts.empty:
        print("[MTS Excel] No data — skipping.")
        return
    
    today = pd.Timestamp.now().normalize()
    
    # ── Formats ──────────────────────────────────────────────
    hdr  = workbook.add_format({"bold":True,"font_name":"Arial","font_size":9,"bg_color":"#1F3864","font_color":"#FFFFFF","border":1,"text_wrap":True,"valign":"vcenter"})
    bfmt = workbook.add_format({"font_name":"Arial","font_size":9,"border":1})
    afmt = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"bg_color":"#F2F2F2"})
    nf   = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0"})
    nfa  = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"#,##0","bg_color":"#F2F2F2"})
    mf   = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"$#,##0"})
    mfa  = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"$#,##0","bg_color":"#F2F2F2"})
    df_  = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"yyyy-mm-dd"})
    dfa  = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"yyyy-mm-dd","bg_color":"#F2F2F2"})
    albl = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"bg_color":"#FFC000","italic":True,"bold":True})
    amny = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"bg_color":"#FFC000","italic":True,"bold":True,"num_format":"$#,##0"})
    pgrn = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.0%","font_color":"#006100"})
    pred = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"num_format":"0.0%","font_color":"#C00000"})
    sfmt = workbook.add_format({"bold":True,"font_name":"Arial","font_size":11,"font_color":"#1F3864","bottom":2})
    tlbl = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"bold":True,"bg_color":"#D9E2F3"})
    tmny = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"bold":True,"bg_color":"#D9E2F3","num_format":"$#,##0"})
    tnum = workbook.add_format({"font_name":"Arial","font_size":9,"border":1,"bold":True,"bg_color":"#D9E2F3","num_format":"#,##0"})
    
    # ── Prepare NIH data ─────────────────────────────────────
    df_nih = df_mts[df_mts["is_nih"]].copy().sort_values("record_date").reset_index(drop=True)
    if df_nih.empty:
        df_nih = df_mts.copy().sort_values("record_date").reset_index(drop=True)
    
    annual_agg = (
        df_nih.groupby("record_calendar_year")["current_month_net_outly_amt"]
        .agg(total="sum", count="count").reset_index()
        .sort_values("record_calendar_year")
    )
    annual_agg["record_calendar_year"] = annual_agg["record_calendar_year"].astype(int)
    
    monthly_agg = (
        df_nih.groupby(["record_calendar_year","record_calendar_month"])["current_month_net_outly_amt"]
        .sum().reset_index()
        .sort_values(["record_calendar_year","record_calendar_month"])
    )
    monthly_agg["record_calendar_year"] = monthly_agg["record_calendar_year"].astype(int)
    monthly_agg["record_calendar_month"] = monthly_agg["record_calendar_month"].astype(int)
    
    # ══════════════════════════════════════════════════════════
    # TAB 1: MTS Database
    # ══════════════════════════════════════════════════════════
    ws_db = ws_dict.get("MTS Database", workbook.add_worksheet("MTS Database")) if ws_dict else workbook.add_worksheet("MTS Database")
    
    db_cols = [
        ("Record Date",              "record_date",                  18, "date"),
        ("Agency",                   "classification_desc",          40, "text"),
        ("Parent Agency",            "parent_desc",                  40, "text"),
        ("Monthly Net Outlays ($)", "current_month_net_outly_amt",  22, "money"),
        ("FYTD Net Outlays ($)",    "current_fytd_net_outly_amt",   22, "money"),
        ("Prior FYTD Outlays ($)",  "prior_fytd_net_outly_amt",     22, "money"),
        ("Fiscal Year",              "record_fiscal_year",           12, "num"),
        ("Calendar Year",            "record_calendar_year",         14, "num"),
        ("Calendar Month",           "record_calendar_month",        14, "num"),
    ]
    
    for ci, (label, _, width, _) in enumerate(db_cols):
        ws_db.write(0, ci, label, hdr)
        ws_db.set_column(ci, ci, width)
    
    for ri, (_, row_data) in enumerate(df_nih.iterrows()):
        er = ri + 1
        ia = ri % 2 == 1
        for ci, (_, field, _, dtype) in enumerate(db_cols):
            val = row_data.get(field, "")
            if dtype == "date":
                if pd.notna(val):
                    ws_db.write_datetime(er, ci, val.to_pydatetime(), dfa if ia else df_)
                else:
                    ws_db.write_blank(er, ci, None, afmt if ia else bfmt)
            elif dtype == "money":
                ws_db.write_number(er, ci, float(val) if pd.notna(val) else 0, mfa if ia else mf)
            elif dtype == "num":
                ws_db.write_number(er, ci, int(val) if pd.notna(val) else 0, nfa if ia else nf)
            else:
                ws_db.write_string(er, ci, str(val) if pd.notna(val) else "", afmt if ia else bfmt)
    
    db_rows = len(df_nih)
    ws_db.autofilter(0, 0, db_rows, len(db_cols) - 1)
    ws_db.freeze_panes(1, 0)
    
    DB = "'MTS Database'"
    DR1 = 2
    DR2 = db_rows + 1
    
    print(f"[MTS Excel] Database: {db_rows} rows")
    
    # ══════════════════════════════════════════════════════════
    # TAB 2: MTS Summary Statistics
    # ══════════════════════════════════════════════════════════
    ws = ws_dict.get("MTS Summary Statistics", workbook.add_worksheet("MTS Summary Statistics")) if ws_dict else workbook.add_worksheet("MTS Summary Statistics")
    ws.set_column(0, 0, 20)
    ws.set_column(1, 4, 22)
    
    years = sorted(annual_agg["record_calendar_year"].tolist())
    current_year = today.year
    
    row = 0
    ws.merge_range(row, 0, row, 4, "NIH Monthly Treasury Outlays — Annual Summary", sfmt)
    row += 1
    for ci, h in enumerate(["Year", "NIH Outlays ($)", "Months Reported"]):
        ws.write(row, ci, h, hdr)
    row += 1
    
    ann_start = row
    for yi, yr in enumerate(years):
        ia = yi % 2 == 1
        ws.write_string(row, 0, str(yr), afmt if ia else bfmt)
        ws.write_formula(row, 1,
            f"=SUMPRODUCT(({DB}!$H${DR1}:$H${DR2}={yr})*({DB}!$D${DR1}:$D${DR2}))",
            mfa if ia else mf)
        ws.write_formula(row, 2,
            f"=COUNTIF({DB}!$H${DR1}:$H${DR2},{yr})",
            nfa if ia else nf)
        row += 1
    
    has_ann = current_year in years
    if has_ann:
        cy_row = ann_start + years.index(current_year)
        ws.write_string(row, 0, f"{current_year} Ann.", albl)
        ws.write_formula(row, 1,
            f"=ROUND(B{cy_row+1}*12/MONTH(EOMONTH(TODAY(),-1)),0)", amny)
        ws.write_string(row, 2, "Annualized", albl)
        row += 1
    
    # Rolling metrics
    row += 2
    ws.merge_range(row, 0, row, 4, "NIH Outlays — Rolling Metrics", sfmt)
    row += 1
    
    last_cm = pd.Timestamp((today.replace(day=1) - timedelta(days=1)).replace(day=1))
    t12e = last_cm + pd.offsets.MonthEnd(0)
    t12s = (t12e - pd.DateOffset(months=11)).replace(day=1)
    t12pe = t12s - timedelta(days=1)
    t12ps = (t12pe - pd.DateOffset(months=11)).replace(day=1)
    
    df_w = (df_nih.groupby(["record_calendar_year","record_calendar_month"])
            .agg(out=("current_month_net_outly_amt","sum"), dt=("record_date","max"))
            .reset_index())
    
    def psum(s, e): return df_w.loc[(df_w["dt"]>=s)&(df_w["dt"]<=e), "out"].sum()
    def ppd(s, e):
        v = psum(s, e); return v / max((e-s).days+1, 1)
    
    t12m = psum(t12s, t12e); t12mp = psum(t12ps, t12pe)
    t12pct = (t12m/t12mp-1) if t12mp else None
    yoy = (ppd(t12s,t12e)/ppd(t12ps,t12pe)-1) if ppd(t12ps,t12pe) else None
    me = last_cm + pd.offsets.MonthEnd(0); ms = last_cm
    mpe = ms - timedelta(days=1); mps = pd.Timestamp(mpe).replace(day=1)
    mom = (ppd(ms,me)/ppd(mps,mpe)-1) if ppd(mps,mpe) else None
    
    for ci, h in enumerate(["Metric", "Value", f"Anchor: {last_cm.strftime('%b %Y')}"]):
        ws.write(row, ci, h, hdr)
    row += 1
    for label, val, mt in [
        ("T12M Outlays ($)", t12m, "m"), ("T12M Prior ($)", t12mp, "m"),
        ("T12M vs Prior %", t12pct, "p"), ("YoY % (per-day)", yoy, "p"),
        ("MoM % (per-day)", mom, "p"),
    ]:
        ws.write_string(row, 0, label, bfmt)
        if val is None: ws.write_string(row, 1, "N/A", bfmt)
        elif mt == "m": ws.write_number(row, 1, float(val), mf)
        else: ws.write_number(row, 1, float(val), pgrn if val >= 0 else pred)
        ws.write_blank(row, 2, None, bfmt)
        row += 1
    
    # Monthly detail
    row += 2
    ws.merge_range(row, 0, row, 4, "NIH Outlays — Monthly Detail", sfmt)
    row += 1
    for ci, h in enumerate(["Month", "NIH Outlays ($)", "Days", "Outlays/Day ($)"]):
        ws.write(row, ci, h, hdr)
    row += 1
    
    m_start = row
    cursor = pd.Timestamp(start_date_str).replace(day=1)
    m_end_dt = pd.Timestamp(end_date_str)
    months_list = []
    while cursor <= m_end_dt:
        months_list.append(cursor)
        cursor += pd.DateOffset(months=1)
    
    for mi, m in enumerate(months_list):
        ia = mi % 2 == 1
        yr, mo = m.year, m.month
        er1 = row + 1
        ws.write_string(row, 0, m.strftime("%b %Y"), afmt if ia else bfmt)
        ws.write_formula(row, 1,
            f"=SUMPRODUCT(({DB}!$H${DR1}:$H${DR2}={yr})"
            f"*({DB}!$I${DR1}:$I${DR2}={mo})"
            f"*({DB}!$D${DR1}:$D${DR2}))",
            mfa if ia else mf)
        ws.write_number(row, 2, calendar.monthrange(yr, mo)[1], nfa if ia else nf)
        ws.write_formula(row, 3, f"=IF(C{er1}=0,0,B{er1}/C{er1})", mfa if ia else mf)
        row += 1
    
    m_end_row = row - 1
    ws.write_string(row, 0, "Total", tlbl)
    ws.write_formula(row, 1, f"=SUM(B{m_start+1}:B{m_end_row+1})", tmny)
    ws.write_formula(row, 2, f"=SUM(C{m_start+1}:C{m_end_row+1})", tnum)
    ws.write_formula(row, 3, f"=IF(C{row+1}=0,0,B{row+1}/C{row+1})", tmny)
    
    print(f"[MTS Excel] Summary: {len(years)} yrs, {len(months_list)} months")
    
    # ══════════════════════════════════════════════════════════
    # TAB 3: MTS Charts (FIXED — uses hidden helper sheet)
    # ══════════════════════════════════════════════════════════
    ws_h = ws_dict.get("_MTS_ChartData", workbook.add_worksheet("_MTS_ChartData")) if ws_dict else workbook.add_worksheet("_MTS_ChartData")
    ws_h.hide()
    ws_c = ws_dict.get("MTS Charts", workbook.add_worksheet("MTS Charts")) if ws_dict else workbook.add_worksheet("MTS Charts")
    
    DARK_GREY = "#8C8C8C"  # RGB(140, 140, 140)
    LIGHT_BLUE = "#1E9BD7"  # RGB(30, 155, 215)
    DARK_BLUE = "#1F3864"
    
    # ── Yearly chart data (convert $ to $B) ──
    chart_labels = []
    chart_values = []
    for yr in years:
        yr_total = annual_agg.loc[annual_agg["record_calendar_year"] == yr, "total"]
        chart_labels.append(str(yr))
        chart_values.append(float(yr_total.iloc[0]) / 1_000_000_000.0 if len(yr_total) else 0.0)
    
    last_cm_month = last_cm.month
    if has_ann:
        cy_val = chart_values[years.index(current_year)]
        ann_val = round(cy_val * 12 / last_cm_month, 1) if last_cm_month > 0 else 0
        chart_labels.append(f"{current_year} Ann.")
        chart_values.append(ann_val)
    
    nb_ = len(chart_labels)
    
    for i in range(nb_):
        ws_h.write_string(i, 0, chart_labels[i])
        ws_h.write_number(i, 1, chart_values[i])
    
    # Per-bar colors: dark grey historical, light blue for last 2 (or 1)
    bc = 2 if has_ann else 1
    yearly_points = []
    for i in range(nb_):
        color = LIGHT_BLUE if i >= nb_ - bc else DARK_GREY
        yearly_points.append({"fill": {"color": color}, "border": {"color": color}})
    
    # YEARLY CHART
    ch1 = workbook.add_chart({"type": "column"})
    ch1.add_series({
        "name":       "NIH Outlays",
        "categories": ["_MTS_ChartData", 0, 0, nb_-1, 0],
        "values":     ["_MTS_ChartData", 0, 1, nb_-1, 1],
        "fill":       {"color": DARK_GREY},
        "gap":        80,
        "points":     yearly_points,
        "data_labels": {
            "value": True,
            "num_format": "$#,##0.0",
            "font": {"name": "Arial", "size": 8, "bold": True, "color": DARK_BLUE},
            "position": "outside_end",
        },
    })
    ch1.set_title({"name": "NIH Treasury Outlays ($B): Yearly",
        "name_font": {"name":"Arial","size":12,"bold":True,"color":DARK_BLUE}})
    ch1.set_x_axis({"num_font": {"name":"Arial","size":8}})
    ch1.set_y_axis({"num_format": "$#,##0", "major_gridlines": {"visible": False}})
    ch1.set_legend({"none": True})
    ch1.set_chartarea({"border": {"none": True}})
    ch1.set_plotarea({"border": {"none": True}})
    ch1.set_size({"width": 720, "height": 420})
    ws_c.insert_chart("A1", ch1)
    
    # ── Monthly chart data (last 24 months only, convert to $B) ──
    chart_months = months_list[-24:] if len(months_list) > 24 else months_list
    n_chart_months = len(chart_months)
    for i, m in enumerate(chart_months):
        ws_h.write_string(i, 3, m.strftime("%b %y"))
        match = monthly_agg[
            (monthly_agg["record_calendar_year"] == m.year) &
            (monthly_agg["record_calendar_month"] == m.month)
        ]
        val = float(match["current_month_net_outly_amt"].iloc[0]) / 1_000_000_000.0 if len(match) else 0.0
        ws_h.write_number(i, 4, val)
    
    monthly_points = []
    for i in range(n_chart_months):
        color = LIGHT_BLUE if i == n_chart_months - 1 else DARK_GREY
        monthly_points.append({"fill": {"color": color}, "border": {"color": color}})
    
    # MONTHLY CHART
    ch2 = workbook.add_chart({"type": "column"})
    ch2.add_series({
        "name":       "NIH Outlays",
        "categories": ["_MTS_ChartData", 0, 3, n_chart_months-1, 3],
        "values":     ["_MTS_ChartData", 0, 4, n_chart_months-1, 4],
        "fill":       {"color": DARK_GREY},
        "gap":        60,
        "points":     monthly_points,
        "data_labels": {
            "value": True,
            "num_format": "$#,##0.0",
            "font": {"name":"Arial","size":7,"bold":True,"color":DARK_BLUE},
            "position": "outside_end",
        },
    })
    ch2.set_title({"name": "NIH Treasury Outlays ($B): Monthly",
        "name_font": {"name":"Arial","size":12,"bold":True,"color":DARK_BLUE}})
    ch2.set_x_axis({"num_font": {"name":"Arial","size":7}, "label_position": "low"})
    ch2.set_y_axis({"num_format": "$#,##0.0", "major_gridlines": {"visible": False}})
    ch2.set_legend({"none": True})
    ch2.set_chartarea({"border": {"none": True}})
    ch2.set_plotarea({"border": {"none": True}})
    ch2.set_size({"width": 720, "height": 400})
    ws_c.insert_chart("A23", ch2)
    
    print("[MTS Excel] Charts: 2 charts. Module complete.")

print("NIH builder ready")


NIH builder ready


In [16]:
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML, clear_output
import xlsxwriter
from datetime import date
import traceback as _tb

DARK_NAV = "#1F3864"
LIGHT_BLU = "#BDD7EE"

# ── Separator Tab ──
def _make_separator(wb, name, color=DARK_NAV):
    ws = wb.add_worksheet(name)
    ws.set_tab_color(color)
    ws.hide_gridlines(2)
    fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":14,
        "font_color":"#FFFFFF","bg_color":color,"align":"center","valign":"vcenter"})
    ws.merge_range("A1:E3", name.strip().replace("─","").strip(), fmt)
    ws.set_column(0, 4, 20)
    return ws

# ── Cover Sheet ──
def build_cover(wb, start_date, end_date, audit_log, modules_selected):
    ws = wb.add_worksheet("Cover")
    ws.set_tab_color(DARK_NAV)
    ws.hide_gridlines(2)
    ws.set_column(0, 0, 5)
    ws.set_column(1, 1, 35)
    ws.set_column(2, 2, 55)
    ws.set_column(3, 3, 25)

    # Title area
    title_fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":22,
        "font_color":"#FFFFFF","bg_color":DARK_NAV,"align":"center","valign":"vcenter"})
    sub_fmt = wb.add_format({"font_name":"Arial","font_size":11,
        "font_color":"#FFFFFF","bg_color":DARK_NAV,"align":"center","valign":"vcenter"})
    ws.merge_range("A1:D3", "Biopharma Monthly Intelligence Report", title_fmt)
    ws.merge_range("A4:D4",
        f"Date Range: {start_date}  to  {end_date}   |   Generated: {date.today().strftime('%B %d, %Y')}",
        sub_fmt)
    ws.merge_range("A5:D5", "", wb.add_format({"bg_color":DARK_NAV}))

    # Audit log section
    sec_fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":13,
        "font_color":DARK_NAV,"bottom":2,"bottom_color":DARK_NAV})
    hdr_fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,
        "bg_color":DARK_NAV,"font_color":"#FFFFFF","border":1,"align":"center"})
    ok_fmt = wb.add_format({"font_name":"Arial","font_size":10,"border":1,
        "font_color":"#006100","bold":True,"align":"center","bg_color":"#E2EFDA"})
    fail_fmt = wb.add_format({"font_name":"Arial","font_size":10,"border":1,
        "font_color":"#C00000","bold":True,"align":"center","bg_color":"#FCE4EC"})
    skip_fmt = wb.add_format({"font_name":"Arial","font_size":10,"border":1,
        "font_color":"#666666","italic":True,"align":"center","bg_color":"#F2F2F2"})
    cell_fmt = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"valign":"vcenter"})

    r = 7
    ws.merge_range(f"B{r}:D{r}", "Module Status", sec_fmt); r += 1
    ws.write(r-1, 1, "Module", hdr_fmt)
    ws.write(r-1, 2, "Details", hdr_fmt)
    ws.write(r-1, 3, "Status", hdr_fmt)
    for entry in audit_log:
        ws.write_blank(r-1, 0, None)
        ws.write(r-1, 1, entry["module"], cell_fmt)
        ws.write(r-1, 2, entry.get("detail", ""), cell_fmt)
        if entry["status"] == "OK":
            ws.write(r-1, 3, "✓ OK", ok_fmt)
        elif entry["status"] == "FAILED":
            ws.write(r-1, 3, "✗ FAILED", fail_fmt)
        else:
            ws.write(r-1, 3, "— SKIPPED", skip_fmt)
        r += 1

    # Table of contents
    r += 1
    ws.merge_range(f"B{r}:D{r}", "Report Contents", sec_fmt); r += 1
    toc_fmt = wb.add_format({"font_name":"Arial","font_size":10,"font_color":DARK_NAV,
        "underline":True})
    sections = [("Charts", ["CT Charts", "FDA Charts", "MTS Charts"]),
                ("Summary", ["Summary Statistics", "Summary Statistics - Enrollment",
                             "FDA Approval Summary", "MTS Summary Statistics"]),
                ("Database", ["Trial Database", "Novel Approval Database",
                              "Biosimilar Approval Database", "MTS Database"])]
    for sec_name, tabs in sections:
        ws.write(r-1, 1, f"  {sec_name}", wb.add_format({"bold":True,"font_name":"Arial",
            "font_size":10,"font_color":DARK_NAV}))
        r += 1
        for tab in tabs:
            ws.write(r-1, 2, tab, toc_fmt)
            r += 1
    return ws

# ── Pre-create all worksheets in desired order ──
def _precreate_tabs(wb, modules_selected):
    """Create worksheets in display order: Cover → Charts → Summary → Database.
    Returns dict of name → worksheet object."""
    ws = {}

    # Cover (created by build_cover later, but we need it first)
    # Actually Cover is built last (needs audit data), but created first for ordering
    # We'll handle this by creating a placeholder

    # Charts section
    _make_separator(wb, "── Charts ──")
    if "ct" in modules_selected:
        ws["CT Charts"] = wb.add_worksheet("CT Charts")
        ws["CT Charts"].set_tab_color(LIGHT_BLU)
    if "fda" in modules_selected:
        ws["FDA Charts"] = wb.add_worksheet("FDA Charts")
        ws["FDA Charts"].set_tab_color(LIGHT_BLU)
    if "nih" in modules_selected:
        ws["_MTS_ChartData"] = wb.add_worksheet("_MTS_ChartData")
        ws["_MTS_ChartData"].hide()
        ws["MTS Charts"] = wb.add_worksheet("MTS Charts")
        ws["MTS Charts"].set_tab_color(LIGHT_BLU)

    # Summary section
    _make_separator(wb, "── Summary ──")
    if "ct" in modules_selected:
        ws["Summary Statistics"] = wb.add_worksheet("Summary Statistics")
        ws["Summary Statistics"].set_tab_color(LIGHT_BLU)
        ws["Summary Statistics - Enrollment"] = wb.add_worksheet("Summary Statistics - Enrollment")
        ws["Summary Statistics - Enrollment"].set_tab_color(LIGHT_BLU)
    if "fda" in modules_selected:
        ws["FDA Approval Summary"] = wb.add_worksheet("FDA Approval Summary")
        ws["FDA Approval Summary"].set_tab_color(LIGHT_BLU)
    if "nih" in modules_selected:
        ws["MTS Summary Statistics"] = wb.add_worksheet("MTS Summary Statistics")
        ws["MTS Summary Statistics"].set_tab_color(LIGHT_BLU)

    # Database section
    _make_separator(wb, "── Database ──")
    if "ct" in modules_selected:
        ws["Trial Database"] = wb.add_worksheet("Trial Database")
        ws["Trial Database"].set_tab_color(LIGHT_BLU)
    if "fda" in modules_selected:
        ws["Novel Approval Database"] = wb.add_worksheet("Novel Approval Database")
        ws["Novel Approval Database"].set_tab_color(LIGHT_BLU)
        ws["Biosimilar Approval Database"] = wb.add_worksheet("Biosimilar Approval Database")
        ws["Biosimilar Approval Database"].set_tab_color(LIGHT_BLU)
    if "nih" in modules_selected:
        ws["MTS Database"] = wb.add_worksheet("MTS Database")
        ws["MTS Database"].set_tab_color(LIGHT_BLU)

    return ws

# ── Orchestrator ──
def run_report(start_date, end_date, modules_selected, progress_cb=None):
    audit_log = []
    results = {}
    pull_date = date.today().strftime("%Y%m%d")
    sd_s = start_date.strftime("%Y%m%d")
    ed_s = end_date.strftime("%Y%m%d")
    fname = f"biopharma_report_{sd_s}_{ed_s}_pulled_{pull_date}.xlsx"

    total_steps = len(modules_selected) * 2 + 2
    step = 0
    def _prog(msg):
        nonlocal step; step += 1
        if progress_cb: progress_cb(step / total_steps, msg)

    # ── Fetch phase ──
    if "ct" in modules_selected:
        _prog("Fetching Clinical Trials...")
        try:
            df_ct = fetch_trials(start_date, end_date)
            results["ct"] = df_ct
            audit_log.append({"module":"Clinical Trials","status":"OK",
                "detail":f"{len(df_ct):,} trials fetched"})
        except Exception as e:
            audit_log.append({"module":"Clinical Trials","status":"FAILED","detail":str(e)})
            print(f"CT fetch error: {e}"); _tb.print_exc()
    else:
        audit_log.append({"module":"Clinical Trials","status":"SKIPPED","detail":"Not selected"})
        _prog("Clinical Trials skipped")

    if "fda" in modules_selected:
        _prog("Fetching FDA Approvals...")
        try:
            novel, biosim, cder_date, cber_date, biosim_date = fetch_all_fda(start_date, end_date)
            results["fda"] = (novel, biosim, cder_date, cber_date, biosim_date)
            audit_log.append({"module":"FDA Approvals","status":"OK",
                "detail":f"{len(novel)} novel, {len(biosim)} biosimilar entries"})
        except Exception as e:
            audit_log.append({"module":"FDA Approvals","status":"FAILED","detail":str(e)})
            print(f"FDA fetch error: {e}"); _tb.print_exc()
    else:
        audit_log.append({"module":"FDA Approvals","status":"SKIPPED","detail":"Not selected"})
        _prog("FDA Approvals skipped")

    if "nih" in modules_selected:
        _prog("Fetching NIH Outlays...")
        try:
            history_start = (pd.Timestamp(start_date) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
            df_nih = fetch_mts_outlays(history_start, end_date.strftime("%Y-%m-%d"))
            results["nih"] = df_nih
            nih_count = df_nih[df_nih["is_nih"]]["record_date"].nunique() if not df_nih.empty else 0
            audit_log.append({"module":"NIH Outlays","status":"OK",
                "detail":f"{nih_count} NIH monthly records"})
        except Exception as e:
            audit_log.append({"module":"NIH Outlays","status":"FAILED","detail":str(e)})
            print(f"NIH fetch error: {e}"); _tb.print_exc()
    else:
        audit_log.append({"module":"NIH Outlays","status":"SKIPPED","detail":"Not selected"})
        _prog("NIH Outlays skipped")

    # ── Build phase ──
    _prog("Creating workbook structure...")
    wb = xlsxwriter.Workbook(fname, {"nan_inf_to_errors": False})

    # Create Cover first (for tab ordering) — content written last
    cover_ws_placeholder = wb.add_worksheet("Cover")
    cover_ws_placeholder.set_tab_color(DARK_NAV)

    # Pre-create all module tabs in desired order
    ws_dict = _precreate_tabs(wb, [k for k in ["ct","fda","nih"] if k in results])

    if "ct" in results:
        _prog("Building Clinical Trials tabs...")
        try:
            build_ct_excel(wb, results["ct"], start_date, end_date, ws_dict=ws_dict)
        except Exception as e:
            for entry in audit_log:
                if entry["module"] == "Clinical Trials" and entry["status"] == "OK":
                    entry["status"] = "FAILED"
                    entry["detail"] += f" | Build error: {e}"
            print(f"CT build error: {e}"); _tb.print_exc()

    if "fda" in results:
        _prog("Building FDA Approval tabs...")
        try:
            novel, biosim, cder_date, cber_date, biosim_date = results["fda"]
            build_fda_excel(wb, novel, biosim, start_date, end_date,
                           cder_date, cber_date, biosim_date, ws_dict=ws_dict)
        except Exception as e:
            for entry in audit_log:
                if entry["module"] == "FDA Approvals" and entry["status"] == "OK":
                    entry["status"] = "FAILED"
                    entry["detail"] += f" | Build error: {e}"
            print(f"FDA build error: {e}"); _tb.print_exc()

    if "nih" in results:
        _prog("Building NIH Outlay tabs...")
        try:
            build_mts_excel(wb, results["nih"],
                           start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d"),
                           ws_dict=ws_dict)
        except Exception as e:
            for entry in audit_log:
                if entry["module"] == "NIH Outlays" and entry["status"] == "OK":
                    entry["status"] = "FAILED"
                    entry["detail"] += f" | Build error: {e}"
            print(f"NIH build error: {e}"); _tb.print_exc()

    # Write cover content (created first for position, written last for audit data)
    cover_ws_placeholder.hide_gridlines(2)
    cover_ws_placeholder.set_column(0, 0, 5)
    cover_ws_placeholder.set_column(1, 1, 35)
    cover_ws_placeholder.set_column(2, 2, 55)
    cover_ws_placeholder.set_column(3, 3, 25)

    # Title banner
    title_fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":22,
        "font_color":"#FFFFFF","bg_color":DARK_NAV,"align":"center","valign":"vcenter"})
    sub_fmt = wb.add_format({"font_name":"Arial","font_size":11,
        "font_color":"#FFFFFF","bg_color":DARK_NAV,"align":"center","valign":"vcenter"})
    bar_fmt = wb.add_format({"bg_color":DARK_NAV})
    cover_ws_placeholder.merge_range("A1:D3", "Biopharma Monthly Intelligence Report", title_fmt)
    cover_ws_placeholder.merge_range("A4:D4",
        f"Date Range: {start_date.strftime('%b %d, %Y')}  to  {end_date.strftime('%b %d, %Y')}   |   Generated: {date.today().strftime('%B %d, %Y')}",
        sub_fmt)
    cover_ws_placeholder.merge_range("A5:D5", "", bar_fmt)

    # Audit log
    sec_fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":13,
        "font_color":DARK_NAV,"bottom":2,"bottom_color":DARK_NAV})
    hdr_fmt = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,
        "bg_color":DARK_NAV,"font_color":"#FFFFFF","border":1,"align":"center"})
    ok_fmt = wb.add_format({"font_name":"Arial","font_size":10,"border":1,
        "font_color":"#006100","bold":True,"align":"center","bg_color":"#E2EFDA"})
    fail_fmt = wb.add_format({"font_name":"Arial","font_size":10,"border":1,
        "font_color":"#C00000","bold":True,"align":"center","bg_color":"#FCE4EC"})
    skip_fmt = wb.add_format({"font_name":"Arial","font_size":10,"border":1,
        "font_color":"#666666","italic":True,"align":"center","bg_color":"#F2F2F2"})
    cell_fmt = wb.add_format({"font_name":"Arial","font_size":9,"border":1,"valign":"vcenter"})
    toc_fmt = wb.add_format({"font_name":"Arial","font_size":10,"font_color":DARK_NAV})
    toc_sec = wb.add_format({"bold":True,"font_name":"Arial","font_size":10,"font_color":DARK_NAV})

    r = 6
    cover_ws_placeholder.merge_range(f"B{r+1}:D{r+1}", "Module Status", sec_fmt); r += 2
    cover_ws_placeholder.write(r-1, 1, "Module", hdr_fmt)
    cover_ws_placeholder.write(r-1, 2, "Details", hdr_fmt)
    cover_ws_placeholder.write(r-1, 3, "Status", hdr_fmt); r += 1
    for entry in audit_log:
        cover_ws_placeholder.write(r-1, 1, entry["module"], cell_fmt)
        cover_ws_placeholder.write(r-1, 2, entry.get("detail", ""), cell_fmt)
        if entry["status"] == "OK":
            cover_ws_placeholder.write(r-1, 3, "OK", ok_fmt)
        elif entry["status"] == "FAILED":
            cover_ws_placeholder.write(r-1, 3, "FAILED", fail_fmt)
        else:
            cover_ws_placeholder.write(r-1, 3, "SKIPPED", skip_fmt)
        r += 1

    r += 1
    cover_ws_placeholder.merge_range(f"B{r+1}:D{r+1}", "Report Contents", sec_fmt); r += 2
    sections = [("Charts", ["CT Charts", "FDA Charts", "MTS Charts"]),
                ("Summary", ["Summary Statistics", "Summary Statistics - Enrollment",
                             "FDA Approval Summary", "MTS Summary Statistics"]),
                ("Database", ["Trial Database", "Novel Approval Database",
                              "Biosimilar Approval Database", "MTS Database"])]
    for sec_name, tabs in sections:
        cover_ws_placeholder.write(r-1, 1, sec_name, toc_sec); r += 1
        for tab in tabs:
            if tab in ws_dict:
                cover_ws_placeholder.write(r-1, 2, f"  {tab}", toc_fmt); r += 1

    wb.close()
    _prog("Done!")
    return fname, audit_log

# ── UI ──
start_picker = widgets.DatePicker(description="Start Date:", value=date(2020, 1, 1),
    style={"description_width":"90px"})
end_picker = widgets.DatePicker(description="End Date:", value=date(2026, 2, 28),
    style={"description_width":"90px"})

cb_ct = widgets.Checkbox(value=True, description="Clinical Trials", indent=False)
cb_fda = widgets.Checkbox(value=True, description="FDA Approvals", indent=False)
cb_nih = widgets.Checkbox(value=True, description="NIH Outlays", indent=False)

run_btn = widgets.Button(description="Run Report", button_style="success", icon="play",
    layout=widgets.Layout(width="140px", height="36px"))
clear_btn = widgets.Button(description="Clear All Cache", button_style="warning", icon="trash",
    layout=widgets.Layout(width="160px", height="36px"))
progress = widgets.FloatProgress(value=0, min=0, max=1.0, description="Progress:",
    bar_style="info", layout=widgets.Layout(width="500px"))
status_label = widgets.HTML(value="Ready")
output_area = widgets.Output()

def _progress_cb(pct, msg):
    progress.value = pct
    status_label.value = f"<i>{msg}</i>"

def on_run(b):
    with output_area:
        clear_output(wait=True)
        if not start_picker.value or not end_picker.value:
            display(HTML("<b style='color:red'>Please select both dates.</b>"))
            return
        progress.value = 0

        sd = start_picker.value
        ed = end_picker.value
        if isinstance(sd, _dt.date) and not isinstance(sd, _dt.datetime):
            sd = _dt.date(sd.year, sd.month, sd.day)
        if isinstance(ed, _dt.date) and not isinstance(ed, _dt.datetime):
            ed = _dt.date(ed.year, ed.month, ed.day)

        selected = []
        if cb_ct.value: selected.append("ct")
        if cb_fda.value: selected.append("fda")
        if cb_nih.value: selected.append("nih")

        if not selected:
            display(HTML("<b style='color:red'>Select at least one module.</b>"))
            return

        try:
            fname, audit_log = run_report(sd, ed, selected, _progress_cb)

            ok = sum(1 for a in audit_log if a["status"]=="OK")
            fail = sum(1 for a in audit_log if a["status"]=="FAILED")
            skip = sum(1 for a in audit_log if a["status"]=="SKIPPED")

            display(HTML(
                '<div style="display:flex;gap:16px;margin:12px 0">'
                f'<div style="background:#1F3864;color:#fff;padding:14px 24px;border-radius:8px;text-align:center;flex:1">'
                f'<div style="font-size:28px;font-weight:bold">{ok}</div><div style="font-size:12px">Modules OK</div></div>'
                f'<div style="background:{"#C00000" if fail else "#1F3864"};color:#fff;padding:14px 24px;border-radius:8px;text-align:center;flex:1">'
                f'<div style="font-size:28px;font-weight:bold">{fail}</div><div style="font-size:12px">Failed</div></div>'
                f'<div style="background:#666;color:#fff;padding:14px 24px;border-radius:8px;text-align:center;flex:1">'
                f'<div style="font-size:28px;font-weight:bold">{skip}</div><div style="font-size:12px">Skipped</div></div>'
                '</div>'
            ))

            for entry in audit_log:
                icon = "OK" if entry["status"]=="OK" else ("FAIL" if entry["status"]=="FAILED" else "SKIP")
                color = "#006100" if entry["status"]=="OK" else ("#C00000" if entry["status"]=="FAILED" else "#666")
                display(HTML(f'<span style="color:{color};font-weight:bold">[{icon}]</span> '
                             f'<b>{entry["module"]}</b>: {entry.get("detail","")}'))

            display(HTML("<h4>Download</h4>"))
            display(FileLink(fname, result_html_prefix="Click to download: "))
            status_label.value = "<b style='color:green'>Report ready!</b>"

        except Exception as e:
            progress.value = 0
            status_label.value = f"<b style='color:red'>Error: {e}</b>"
            _tb.print_exc()

def on_clear(b):
    with output_area:
        clear_output(wait=True)
        try: clear_cache()
        except: pass
        try: clear_fda_cache()
        except: pass
        try: clear_mts_cache()
        except: pass
        status_label.value = "<b style='color:orange'>All caches cleared.</b>"
        progress.value = 0

run_btn.on_click(on_run)
clear_btn.on_click(on_clear)

display(widgets.VBox([
    widgets.HTML("<h2 style='color:#1F3864'>Biopharma Monthly Intelligence Report</h2><hr>"),
    widgets.HTML("<b>Date Range</b>"),
    widgets.HBox([start_picker, end_picker]),
    widgets.HTML("<b>Modules</b>"),
    widgets.HBox([cb_ct, cb_fda, cb_nih]),
    widgets.HBox([run_btn, clear_btn]),
    progress, status_label, output_area,
]))
